# Phase 7 -- Prior Case Retrieval: all runs in one notebook

**The problem in three facts.** (1) Queries and candidates are whole judgments -- thousands of words, far longer than
the 512-1,024 tokens a Phase-1 model reads. (2) A query cites a prior case for one *part* of its reasoning, so two
judgments match on a few passages, not on average. (3) The metric is micro-F1@7: every query returns exactly 7 cases, so
precision at the top of the ranking is what counts.

**The approach (same for every run, so runs differ only in model + technique)**
1. **Chunk** every judgment into 360-word pieces (at most 12, spread over the judgment, first and last always kept).
2. **BM25** (own code) over each split's pool: a strong lexical baseline, the source of **hard negatives** (highly ranked
   but not cited), and a second opinion fused with the model at the end.
3. **Evidence-aligned pairs**: for every (query, cited case) the chunk pairs that share the most TF-IDF weight. The model
   is trained on the passages that make one judgment cite the other, not on two random windows.
4. **Dual encoder**: the adapted Phase-1 backbone + attention pooling -> one L2-normalised vector per chunk.
   **InfoNCE** (temperature 0.05) over in-batch candidates + 1 BM25 hard negative per query; other cases the same query
   cites are masked out, so they are never treated as negatives.
5. **Document score** from cached chunk vectors, three ways: *mean* (one vector per judgment), *max-sim* (each query
   chunk finds its best candidate chunk -- late interaction at chunk level, ColBERT-style), *max-pair*.
6. **Fuse**: final = z(model) + w * z(BM25), w in {0, .25, .5, .75, 1}. The aggregation and w are chosen by an
   **honest 2-fold check on dev** (choose on one half of the dev queries, score the other half, both ways).

| Run | Phase-1 model | Technique | Why this pairing | Submit |
|---|---|---|---|---|
| **S1** | D encoder (100M, bidirectional) | **LoRMA** r16 on every attention + MLP layer | a bidirectional encoder is the natural dual encoder; D is the best backbone so far; compulsory method; KD teacher; zero-shot check before training | yes |
| **S2** | C (prefix-LM; full bidirectional attention over the input) | **IA3** (learned scales on keys, values, MLP) | ~0.1% of the weights: the cheapest possible retriever adaptation; tests whether rescaling features is enough to turn an LM into a retriever | yes |
| **S3** | B (local/global decoder) | **QLoRA** (frozen 4-bit NF4 B + LoRA r16) | retrieval must encode every pool document: a 4-bit base cuts memory at the step that dominates cost | yes |
| **S4** | A (Llama-style decoder, 163M) | **Partial (layer-wise) fine-tuning**: top 4 blocks + final norm | causal states need their top layers re-learned to summarise a chunk; lower layers keep Phase-1 legal knowledge | yes |
| **S5** | D encoder | **Prompt tuning** (20 learned input vectors, from real token embeddings) | same backbone as S1 = controlled comparison of the smallest method against LoRMA | yes |
| **S6** | half of D (encoder layers 0,2,..,10) | **Knowledge distillation** (listwise: the student copies S1's in-batch similarity distribution, T = 2) + full fine-tuning | compact retriever: half the layers, ~2x faster encoding of the pool | yes |
| **S7** | -- | **Ensemble**: mean of the z-scored final scores of S1-S6 | different models make different mistakes; no extra training | yes |
| R1 | D encoder | LoRMA, **without** BM25 hard negatives (in-batch only) | ablation: what hard negatives add (dev; optional extra submission) | optional |
| BM25 | none | lexical only | does the neural retriever beat word matching? (dev only) | no |

**How to run** -- inputs: `legallm-tokenizers` (`t1_bpe.json`, `t2_unigram.model`) + the four Phase-1 checkpoint
datasets (A, B, C, D). Secret `HF`. **GPU T4 x2, internet on.** **Save & Run All**: about 4.5-5 h.
Submit `p7_out/S1 ... S7/submission.zip`; download `p7_out/report_files.zip`.

## 1. Inputs: tokenizers and checkpoints

In [ ]:
import csv
import gc
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch

NB_T0 = time.time()
N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p7_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / "ckpts"
CKPT_DIR.mkdir(exist_ok=True)


def step_of(p):
    return int(Path(p).name.split("_step")[1].split("_")[0].split(".")[0])


def _rebuild_from_folder(folder, dest):
    """Kaggle unzips an uploaded checkpoint (a .pt file is a zip archive inside),
    leaving a folder with data.pkl, data/0, version, ... Re-pack it into a .pt."""
    pkl = next(Path(folder).rglob("data.pkl"))
    root = pkl.parent
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    return dest


def _collect(prefix):
    """{name: path} of usable <prefix>_step*.pt checkpoints under /kaggle/input:
    a .pt file; a .zip holding the .pt; the .pt renamed .zip; or a folder that
    Kaggle unzipped from an uploaded checkpoint (re-packed here)."""
    found = {}
    if not KAGGLE_INPUT.exists():
        return found
    for p in KAGGLE_INPUT.rglob(f"{prefix}_step*"):
        stem = p.name[:-3] if p.name.endswith(".pt") else (p.name[:-4] if p.name.endswith(".zip") else p.name)
        dest = CKPT_DIR / (stem + ".pt")
        if p.is_file() and p.suffix == ".pt":
            found.setdefault(p.name, p)
        elif p.is_dir() and any(p.rglob("data.pkl")) and dest.name not in found:
            found[dest.name] = _rebuild_from_folder(p, dest)
            print(f"  re-packed unzipped checkpoint folder {p} -> {dest.name}")
    for z in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                inner = [n for n in names if Path(n).name.startswith(f"{prefix}_step") and n.endswith(".pt")]
                for n in inner:
                    dest = CKPT_DIR / Path(n).name
                    if dest.name not in found:
                        with zf.open(n) as src, open(dest, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        found[dest.name] = dest
                if not inner and z.name.startswith(f"{prefix}_step") and any(n.endswith("data.pkl") for n in names):
                    dest = CKPT_DIR / (z.stem + ".pt")
                    if dest.name not in found:
                        shutil.copy(z, dest)
                        found[dest.name] = dest
        except zipfile.BadZipFile:
            continue
    return found




def checkpoints(prefix):
    found = _collect(prefix)
    assert found, f"No {prefix}_step* checkpoint (.pt, .zip or unzipped folder) under /kaggle/input -- attach it."
    return max(found.values(), key=step_of)



def find_one(name, label):
    hits = sorted(KAGGLE_INPUT.rglob(name)) if KAGGLE_INPUT.exists() else []
    if not hits:                                   # renamed copies, e.g. "t1_bpe (1).json"
        stem, suf = name.rsplit(".", 1)
        hits = sorted(KAGGLE_INPUT.rglob(f"{stem}*.{suf}")) if KAGGLE_INPUT.exists() else []
    if hits:
        return hits[0]
    for z in KAGGLE_INPUT.rglob("*.zip"):          # the file may sit inside a zip
        try:
            with zipfile.ZipFile(z) as zf:
                inner = [n for n in zf.namelist() if Path(n).name == name]
                if inner:
                    dest = WORK_DIR / name
                    with zf.open(inner[0]) as src, open(dest, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    return dest
        except zipfile.BadZipFile:
            continue
    seen = sorted(str(p) for p in KAGGLE_INPUT.rglob("*") if p.suffix in (".json", ".model", ".zip") and "_step" not in p.name)[:40]
    raise AssertionError(f"{name} not found under /kaggle/input -- attach {label}.\nFiles that might be tokenizers: {seen}")


T1_PATH = find_one("t1_bpe.json", "legallm-tokenizers")
T2_PATH = find_one("t2_unigram.model", "legallm-tokenizers")
CKPT = {k: str(checkpoints(k)) for k in ("A", "B", "C", "D")}
for k, p in CKPT.items():
    print(f"{k}: {Path(p).name}")

## 2. Code files (written to /kaggle/working)

In [ ]:
%%writefile /kaggle/working/legal_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample_top_p(logits, temperature, top_p, generator):
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_p < 1.0:
        sorted_p, idx = probs.sort(dim=-1, descending=True)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, idx, sorted_p)
    return torch.multinomial(probs, 1, generator=generator).squeeze(-1)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_enc_len=768, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None, sample=False,
                   temperature=1.0, top_p=1.0, generator=None, start_id=None):
    """Continuations for several prompts at once. Encoder input is right-padded
    (masked); every decoder row starts at the same position, so no decoder
    padding is needed. Greedy by default; `sample=True` gives temperature /
    top-p sampling (used for MBR candidates). Optional repetition penalty over
    generated tokens, n-gram blocking, and early stop once every row has more
    than `target_words` words."""
    model.eval()
    device = model.embed.weight.device
    cfg = model.config
    start_id = DEC_START_ID if start_id is None else start_id
    max_enc_len = min(max_enc_len or cfg["max_enc_len"], cfg["max_enc_len"])
    prompts = [(p[-max_enc_len:] if p else [EOS_ID if EOS_ID is not None else 0]) for p in prompt_id_lists]
    B, P = len(prompts), max(len(p) for p in prompts)
    enc_ids = torch.zeros(B, P, dtype=torch.long, device=device)
    enc_mask = torch.zeros(B, P, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        enc_ids[b, :len(p)] = torch.tensor(p, device=device)
        enc_mask[b, :len(p)] = True
    use_amp = device.type == "cuda" and model.embed.weight.dtype == torch.float32 and getattr(model, "amp", True)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
    with amp:
        enc_out = model.encode(enc_ids, enc_mask)
        cross_kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
        caches = [KVCache(max_new_tokens + 1) for _ in model.dec_blocks]
        tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        logits = model.decoder_step(tok, 0, caches, cross_kvs, enc_mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = _sample_top_p(logits, temperature, top_p, generator) if sample else logits.argmax(dim=-1)
        for b, tk in enumerate(nxt.tolist()):
            generated[b].append(tk)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(len(decode(g).split()) > target_words for g in generated):
            break
        with amp:
            logits = model.decoder_step(nxt[:, None], t + 1, caches, cross_kvs, enc_mask)[:, -1, :].float()
    return generated


# ---------------------------------------------------------------------------
# Scoring helpers (mirror evaluate_phase1.py; BERTScore uses roberta-large)
# ---------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    n = lcs_length(pt, rt)
    if not n:
        return 0.0
    pr, rc = n / len(pt), n / len(rt)
    return 2 * pr * rc / (pr + rc)


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


def score_predictions(pred, truth, ids):
    try:
        import bert_score
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
        import bert_score
    hyps = [pred[i]["text"].strip() for i in ids]
    refs = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


def to_words(toks, n_words=128):
    words = decode(toks).split()[:n_words]
    return " ".join(words) if words else "."


def predict(model, prompt_dict, ids, batch_size=16, desc="predict", **settings):
    """Greedy/sampled continuations -> {id: {"text": 128 words}}. Prompts are
    sorted by length so each batch has similar lengths."""
    from tqdm.auto import tqdm
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


def mbr_pick(candidates):
    """Minimum-Bayes-risk choice: the candidate with the highest average
    ROUGE-L F1 against the other candidates (a consensus pick; no outside
    model involved)."""
    if len(candidates) <= 2:
        return candidates[0]
    best, best_u = 0, -1.0
    for i, c in enumerate(candidates):
        u = sum(rouge_l_f1(c, o) for j, o in enumerate(candidates) if j != i) / (len(candidates) - 1)
        if u > best_u:
            best, best_u = i, u
    return candidates[best]


def predict_mbr(model, prompt_dict, ids, n_samples=7, batch_prompts=8, greedy_settings=None,
                sample_settings=None, seed=1234, desc="mbr"):
    """For each prompt: 1 greedy candidate + n_samples sampled candidates ->
    MBR pick. Candidates of `batch_prompts` prompts are generated together."""
    from tqdm.auto import tqdm
    device = model.embed.weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_prompts), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_prompts]
        greedy = generate_batch(model, [enc[i] for i in chunk], **greedy_settings)
        sampled = generate_batch(model, [enc[i] for i in chunk for _ in range(n_samples)], sample=True,
                                 generator=gen, **sample_settings)
        for k, i in enumerate(chunk):
            cands = [to_words(greedy[k])] + [to_words(t) for t in sampled[k * n_samples:(k + 1) * n_samples]]
            out[i] = {"text": mbr_pick(cands)}
    return {i: out[i] for i in ids}


@torch.no_grad()
def dev_loss_bpb(model, dev_prompts, dev_truth, ids, max_enc_len=768, max_dec_len=512):
    """Continuation loss per token and bits-per-byte (tokenizer-independent)."""
    model.eval()
    device = model.embed.weight.device
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in ids:
        cont_text = " " + dev_truth[i]["text"].strip()
        cont = encode(cont_text)[:max_dec_len]
        ctx = encode(dev_prompts[i]["text"])[-max_enc_len:]
        if not cont or not ctx:
            continue
        enc_ids = torch.tensor([ctx], device=device)
        dec_in = torch.tensor([[DEC_START_ID] + cont[:-1]], device=device)
        tgt = torch.tensor([cont], device=device)
        out = model(enc_ids, dec_in, targets=tgt)[0]
        nll += out[0].item()
        n_tok += len(cont)
        n_bytes += len(cont_text.encode("utf-8"))
    return nll / max(1, n_tok), nll / (math.log(2) * max(1, n_bytes))


def self_test(device):
    """Checks: cached decoding == full forward; decoder is causal; encoder is
    two-way; encoder padding changes nothing; loss sums are correct; batched
    generation == one-at-a-time; Muon orthogonalises."""
    torch.manual_seed(0)
    tiny = dict(MODEL_D_CONFIG, vocab_size=101, d_model=64, n_enc_layer=2, n_dec_layer=2, n_head=4,
                n_kv_head=2, d_head=16, d_ff=96, max_enc_len=64, max_dec_len=64)
    m = build_model(tiny).to(device).eval()
    m.amp = False                                          # exactness checks in fp32 (GPU kernels still used)
    enc = torch.randint(4, 101, (2, 20), device=device)
    dec = torch.randint(4, 101, (2, 12), device=device)
    with torch.no_grad():
        full = m(enc, dec).float()
        enc_out = m.encode(enc)
        kvs = [blk.cross_attn.project_kv(enc_out) for blk in m.dec_blocks]
        caches = [KVCache(12) for _ in m.dec_blocks]
        steps = [m.decoder_step(dec[:, t:t + 1], t, caches, kvs, None) for t in range(12)]
        diff = (full - torch.cat(steps, 1).float()).abs().max().item()
        dec2 = dec.clone()
        dec2[:, 8] = (dec[:, 8] + 1) % 97 + 4
        leak = (m(enc, dec2)[:, :8] - full[:, :8]).abs().max().item()
        enc2 = enc.clone()
        enc2[:, 15] = (enc[:, 15] + 1) % 97 + 4
        two_way = (m.encode(enc2)[:, 2] - enc_out[:, 2]).abs().max().item()
        padded = torch.cat([enc, torch.zeros(2, 5, dtype=torch.long, device=device)], 1)
        mask = torch.cat([torch.ones(2, 20), torch.zeros(2, 5)], 1).to(device)
        pad_diff = (m(padded, dec, enc_mask=mask) - full).abs().max().item()
        tgt = torch.randint(4, 101, (2, 12), device=device)
        tgt[0, 3] = -100
        s = m(enc, dec, targets=tgt)[0]
        ref = F.cross_entropy(full.reshape(-1, 101), tgt.reshape(-1), ignore_index=-100, reduction="sum")
    assert diff < 1e-3, f"cached decoding != full forward ({diff})"
    assert leak < 1e-5, f"decoder sees future tokens ({leak})"
    assert two_way > 1e-5, "encoder is not bidirectional"
    assert pad_diff < 1e-4, f"encoder padding changes the output ({pad_diff})"
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 23, "loss sums wrong"
    pa, pb = enc[0].tolist(), enc[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1)
    alone = [generate_batch(m, [pa], max_new_tokens=8, start_id=1)[0],
             generate_batch(m, [pb], max_new_tokens=8, start_id=1)[0]]
    assert both == alone, "batched generation differs from one-at-a-time"
    g = torch.Generator(device=device)
    g.manual_seed(0)
    smp = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1, sample=True, temperature=0.8, top_p=0.9,
                         generator=g)
    assert len(smp) == 2 and all(len(x) == 8 for x in smp)
    X = newton_schulz5(torch.randn(48, 96, device=device)).float()
    sv = torch.linalg.svdvals(X)
    assert 0.3 < sv.min().item() and sv.max().item() < 1.5, f"Muon orthogonalisation off: {sv.min():.2f}..{sv.max():.2f}"
    print(f"Self-test passed: cached==full {diff:.1e}, causal decoder {leak:.1e}, two-way encoder, "
          f"padding-safe {pad_diff:.1e}, loss sums ok, batched==single, sampling ok, "
          f"Muon singular values {sv.min():.2f}..{sv.max():.2f}.")


# ---------------------------------------------------------------------------
# Muon optimizer (Jordan et al. 2024; update scale from Liu et al. 2025,
# "Muon is Scalable for LLM Training": 0.2 * sqrt(max(rows, cols)) so the
# learning rate and weight decay work on AdamW's scale).
# ---------------------------------------------------------------------------
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + eps)                              # normalise in fp32 (tiny grads survive)
    if G.is_cuda:
        X = X.half()                                       # T4: fast fp16 matmuls, values stay <= 1
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        Bm = b * A + c * (A @ A)
        X = a * X + Bm @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.95, nesterov=True, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom) if group["nesterov"] else buf
                u = newton_schulz5(g, steps=group["ns_steps"]).float()
                p.mul_(1.0 - lr * wd)
                p.add_(u, alpha=-lr * 0.2 * math.sqrt(max(p.shape[0], p.shape[1])))
        return None


def build_optimizers(model, kind, lr, weight_decay):
    muon_params, adam_decay, adam_no_decay = [], [], []
    for name, p in model.named_parameters():
        if p.dim() == 2 and (name.startswith("enc_blocks") or name.startswith("dec_blocks")) and kind == "muon":
            muon_params.append(p)
        elif p.dim() >= 2:
            adam_decay.append(p)                           # embedding (tied output head); all matrices for AdamW
        else:
            adam_no_decay.append(p)                        # norm weights
    adam = torch.optim.AdamW([{"params": adam_decay, "weight_decay": weight_decay},
                              {"params": adam_no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)
    return ([Muon(muon_params, lr=lr, weight_decay=weight_decay)] if muon_params else []) + [adam]


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def lr_factor(step, progress, warmup_steps, cooldown_frac, floor):
    """Warmup-stable-decay on *time*: linear warmup (steps), constant, then a
    1-sqrt cooldown over the last `cooldown_frac` of the training time."""
    w = min(1.0, (step + 1) / max(1, warmup_steps))
    start = 1.0 - cooldown_frac
    if progress < start:
        return w
    r = min(1.0, (progress - start) / max(1e-9, cooldown_frac))
    return w * (floor + (1.0 - floor) * (1.0 - math.sqrt(r)))


def loss_from(out, z_loss_coef):
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + z_loss_coef * s[2]) / n


# ---------------------------------------------------------------------------
# Data: stream of 1024-token windows (round-robin over the cleaned files,
# documents separated by <eos>), byte positions saved for resume. Each GPU
# (rank) starts at a different offset of every file.
# ---------------------------------------------------------------------------
class WindowStream:
    def __init__(self, file_paths, window, rank=0, world=1, start_positions=None):
        self.files = [Path(p) for p in file_paths if Path(p).stat().st_size > 0]
        self.window = window
        self.positions, self.unaligned = {}, set()
        for p in self.files:
            if start_positions and p.name in start_positions:
                self.positions[p.name] = int(start_positions[p.name])
            else:
                self.positions[p.name] = (p.stat().st_size * rank) // world
                if self.positions[p.name] > 0:
                    self.unaligned.add(p.name)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.files]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.window:
                yield buffer[:self.window]
                buffer = buffer[self.window:]

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                if name in self.unaligned:                  # started mid-line: skip to the next line
                    pos += len(f.readline())
                    self.unaligned.discard(name)
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


def make_example_batch(windows, P, Q, device):
    """windows: list of 1024-token lists -> encoder ids [B,P], decoder input
    [B,Q] (start token + shifted continuation), targets [B,Q]."""
    w = torch.tensor(windows, dtype=torch.long)
    enc = w[:, :P]
    tgt = w[:, P:P + Q]
    dec_in = torch.cat([torch.full((w.shape[0], 1), DEC_START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)


def choose_shape(shapes, seed, step, micro):
    """Same (prompt, continuation) split on every GPU for a given micro-step."""
    rnd = random.Random(seed * 1_000_003 + step * 97 + micro)
    return rnd.choices([(s[0], s[1]) for s in shapes], weights=[s[2] for s in shapes])[0]


# ---------------------------------------------------------------------------
# Checkpoints and Kaggle push
# ---------------------------------------------------------------------------
def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file


def prune_old_checkpoints(ckpt_dir, prefix, keep=2):
    ckpts = sorted(Path(ckpt_dir).glob(f"{prefix}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(output_dir, dataset_id, message):
    kdir = Path.home() / ".kaggle"
    if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
        print("Skipping checkpoint push: no Kaggle API token.", flush=True)
        return False
    slug = dataset_id.split("/")[1]
    with open(Path(output_dir) / "dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    try:
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
                                timeout=300).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", str(output_dir), "-m", message, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", str(output_dir), "-r", "zip"])
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print(f"Checkpoint push failed: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}", flush=True)
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).", flush=True)
    return True


def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


# ---------------------------------------------------------------------------
# Trainer (one process per GPU)
# ---------------------------------------------------------------------------
def _probe_micro_batch(model, cfg, device, P, Q):
    """Largest micro-batch (1024-token examples) that fits the memory budget
    on this GPU, with the real optimizers; weights are restored afterwards."""
    saved = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    results = {}
    for bs in cfg["candidate_micro_bs"]:
        optims = build_optimizers(model, cfg["optimizer"], 1e-5, 0.0)
        scaler = make_grad_scaler(True)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        try:
            windows = [[random.randrange(4, 1000) for _ in range(P + Q)] for _ in range(bs)]
            enc, dec_in, tgt = make_example_batch(windows, P, Q, device)
            times = []
            for _ in range(3):
                torch.cuda.synchronize(device)
                t0 = time.time()
                for o in optims:
                    o.zero_grad(set_to_none=True)
                _, total = loss_from(model(enc, dec_in, targets=tgt), 1e-4)
                scaler.scale(total).backward()
                for o in optims:
                    scaler.step(o)
                scaler.update()
                torch.cuda.synchronize(device)
                times.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device) / 1e9
            ok = peak <= cfg["memory_budget_gb"]
            results[bs] = (peak, min(times[1:]), ok)
            if not ok:
                break
        except Exception as exc:                           # noqa: BLE001
            if "out of memory" not in str(exc).lower():
                raise
            results[bs] = (None, None, False)
            break
        finally:
            for o in optims:
                o.zero_grad(set_to_none=True)
            del optims
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    model.load_state_dict(saved)
    return results


def train_main(config_path):
    cfg = json.load(open(config_path))
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=90))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    else:
        rank, world, local_rank = 0, 1, 0
    device = torch.device(f"cuda:{local_rank}") if cuda else torch.device("cpu")
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0

    def log(*a):
        if main:
            print(*a, flush=True)

    load_tokenizer(cfg["tokenizer_path"])
    model_cfg = cfg["model"]
    prefix = cfg["ckpt_prefix"]
    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    session_start, session_deadline = cfg["session_start_epoch"], cfg["session_start_epoch"] + 60 * cfg["time_budget_min"]
    total_seconds = 3600 * cfg["total_train_hours"]
    log(f"[D] world={world} device={device} | budget this session {cfg['time_budget_min']} min, "
        f"whole run {cfg['total_train_hours']} h | optimizer={cfg['optimizer']}")

    torch.manual_seed(cfg["seed"])
    random.seed(cfg["seed"] + rank)
    model = build_model(model_cfg).to(device)
    n_params = count_params(model)
    log(f"[D] parameters: {n_params / 1e6:.1f}M")

    # ---- resume ----------------------------------------------------------
    step, tokens_seen, seconds_before, rank_positions, ckpt = 0, 0, 0.0, None, None
    phase = cfg.get("phase_id", "main")        # a new phase id = continue training with a fresh LR schedule
    phase_start_step = 0
    latest = find_latest_checkpoint(prefix, [ckpt_dir] + cfg.get("resume_search_dirs", []), ckpt_dir) if main else None
    if distributed:
        box = [str(latest) if latest else ""]
        dist.broadcast_object_list(box, src=0)
        latest = Path(box[0]) if box[0] else None
    if latest is not None:
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        assert ckpt["config"] == model_cfg, "Checkpoint config does not match MODEL_D_CONFIG"
        model.load_state_dict(ckpt["model_state"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        same_session = ckpt.get("session_start_epoch") == session_start
        rank_positions = ckpt.get("data_state", {}).get("rank_positions")
        ckpt_phase = ckpt.get("phase_id", "main")
        if ckpt_phase == phase:
            seconds_before = ckpt["seconds_before_session"] if same_session else ckpt["progress_seconds"]
            phase_start_step = ckpt.get("phase_start_step", 0)
        else:
            seconds_before, phase_start_step = 0.0, step
            log(f"[D] new training phase '{phase}' (previous: '{ckpt_phase}') -> fresh LR schedule from step {step:,}")
        log(f"[D] resumed from {latest.name}: step={step:,} tokens={tokens_seen:,} "
            f"phase progress={seconds_before / total_seconds:.1%} ({'same' if same_session else 'new'} session)")
    else:
        log("[D] no checkpoint found -- starting fresh")

    # ---- micro-batch size ---------------------------------------------------
    shapes = cfg["shapes"]
    Pp, Qp = max(((s[0], s[1]) for s in shapes), key=lambda s: s[1])      # longest decoder = most memory
    if cfg.get("fixed_micro_bs") or not cuda:
        micro_bs, sec_micro = cfg.get("fixed_micro_bs") or 2, None
    else:
        res = _probe_micro_batch(model, cfg, device, Pp, Qp)
        fits = [bs for bs, (_, _, ok) in res.items() if ok]
        if not fits:
            model.grad_ckpt = True
            log("[D] nothing fit without activation checkpointing -> turning it on")
            res = _probe_micro_batch(model, cfg, device, Pp, Qp)
            fits = [bs for bs, (_, _, ok) in res.items() if ok]
        for bs, (peak, sec, ok) in res.items():
            log(f"  micro-batch {bs:3d} x 1024 -> " + (f"{peak:5.2f} GB, {sec:.2f}s, {bs * 1024 / sec:,.0f} tok/s/GPU"
                                                      if peak else "OOM") + ("" if ok else "  [no]"))
        best = max(fits) if fits else 0
        if distributed:
            t = torch.tensor([best], device=device)
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
            best = int(t.item())
        assert best > 0, "No micro-batch fits in memory"
        micro_bs, sec_micro = best, res[best][1]
    accum = max(1, round(cfg["tokens_per_step_target"] / (micro_bs * 1024 * world)))
    tokens_per_step = micro_bs * 1024 * world * accum
    log(f"[D] micro-batch {micro_bs} x 1024 tokens x {world} GPU(s) x accum {accum} = {tokens_per_step:,} tokens/step"
        + (f" | probe ~{micro_bs * 1024 * world / sec_micro:,.0f} tok/s" if sec_micro else ""))

    optims = build_optimizers(model, cfg["optimizer"], cfg["peak_lr"], cfg["weight_decay"])
    scaler = make_grad_scaler(cuda)
    if ckpt is not None:
        if ckpt.get("optimizer_kind") == cfg["optimizer"]:
            for o, s in zip(optims, ckpt["optim_states"]):
                o.load_state_dict(s)
            scaler.load_state_dict(ckpt["scaler_state"])
        else:
            log("[D] optimizer changed since the checkpoint -> fresh optimizer state")
        model.grad_ckpt = model.grad_ckpt or ckpt.get("grad_ckpt", False)
        del ckpt
    train_model = model
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        train_model = DDP(model, device_ids=[local_rank] if cuda else None, gradient_as_bucket_view=True)

    positions = rank_positions[rank] if rank_positions and rank < len(rank_positions) else None
    files = sorted(Path(cfg["data_dir"]).glob("clean_*.txt"))
    assert files, "No clean_*.txt files in the data folder"
    stream = WindowStream(files, 1024, rank, world, positions)
    data_iter = iter(stream)
    log(f"[D] training files: {[p.name for p in files]} | shapes (prompt, continuation, weight): {shapes}")

    def save_checkpoint(reason):
        gathered = [None] * world
        if distributed:
            dist.all_gather_object(gathered, dict(stream.positions))
        else:
            gathered = [dict(stream.positions)]
        path = None
        if main:
            now = time.time()
            state = {
                "model_state": model.state_dict(), "optim_states": [o.state_dict() for o in optims],
                "optimizer_kind": cfg["optimizer"], "scaler_state": scaler.state_dict(), "step": step,
                "tokens_seen": tokens_seen, "config": model_cfg, "arch_id": ARCH_ID, "run_id": cfg["run_id"],
                "session_start_epoch": session_start, "seconds_before_session": seconds_before,
                "progress_seconds": seconds_before + (now - session_start), "grad_ckpt": model.grad_ckpt,
                "phase_id": phase, "phase_start_step": phase_start_step,
                "data_state": {"rank_positions": gathered, "world": world},
                "train_cfg": {k: cfg[k] for k in ("peak_lr", "shapes", "optimizer", "total_train_hours")},
                "reason": reason,
            }
            path = ckpt_dir / f"{prefix}_step{step}_tokens{tokens_seen}.pt"
            tmp = path.with_name(path.name + ".tmp")
            torch.save(state, tmp)
            tmp.replace(path)
            prune_old_checkpoints(ckpt_dir, prefix, keep=2)
        if distributed:
            dist.barrier()
        return path

    # ---- loop --------------------------------------------------------------
    train_model.train()
    t_loop, last_ckpt, last_log_t, last_log_tokens = time.time(), time.time(), time.time(), tokens_seen
    start_tokens = tokens_seen
    bad_streak, skipped, stop_reason = 0, 0, "unknown"
    ce_log, n_log = torch.zeros((), device=device), 0
    flags = torch.zeros(3, device=device)
    while True:
        if main:
            now = time.time()
            progress = (seconds_before + (now - session_start)) / total_seconds
            stop = 1.0 if (progress >= 1.0 or now >= session_deadline) else 0.0
            do_ckpt = 1.0 if (now - last_ckpt) / 60 >= cfg["checkpoint_every_min"] else 0.0
            flags = torch.tensor([progress, stop, do_ckpt], device=device)
        if distributed:
            dist.broadcast(flags, src=0)
        progress, stop, do_ckpt = float(flags[0]), bool(flags[1] > 0.5), bool(flags[2] > 0.5)
        if stop:
            stop_reason = "schedule complete" if progress >= 1.0 else "session time budget reached"
            break
        lr = cfg["peak_lr"] * lr_factor(step - phase_start_step, progress, cfg["warmup_steps"], cfg["cooldown_frac"], cfg["min_lr_frac"])
        for o in optims:
            for g in o.param_groups:
                g["lr"] = lr
            o.zero_grad(set_to_none=True)
        for micro in range(accum):
            P, Q = choose_shape(shapes, cfg["seed"], step, micro)
            enc, dec_in, tgt = make_example_batch([next(data_iter) for _ in range(micro_bs)], P, Q, device)
            sync_ctx = train_model.no_sync() if (distributed and micro < accum - 1) else contextlib.nullcontext()
            with sync_ctx:
                ce, total = loss_from(train_model(enc, dec_in, targets=tgt), cfg["z_loss_coef"])
                scaler.scale(total / accum).backward()
            ce_log += ce.detach() / accum
            tokens_seen += micro_bs * 1024 * world
        for o in optims:
            scaler.unscale_(o)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]))
        for o in optims:
            scaler.step(o)
        scaler.update()
        n_log += 1
        step += 1
        if math.isfinite(gnorm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped += 1
            if bad_streak >= 50:
                stop_reason = "50 non-finite steps in a row"
                break
        if distributed and step % cfg["sync_every"] == 0:
            for p in model.parameters():                     # remove any tiny drift between GPUs
                dist.broadcast(p.data, src=0)
        if step % cfg["log_every"] == 0 or step == 1:
            if distributed:
                dist.all_reduce(ce_log)
                ce_log /= world
            now = time.time()
            tps = (tokens_seen - last_log_tokens) / max(1e-6, now - last_log_t)
            log(f"  step {step:6d}  loss={ce_log.item() / max(1, n_log):.4f}  lr={lr:.2e}  gnorm={gnorm:.2f}  "
                f"tokens={tokens_seen:,}  tok/s={tps:,.0f}  elapsed={(now - t_loop) / 60:.1f} min  "
                f"progress={progress:.1%}  [{datetime.datetime.now():%H:%M:%S}]")
            ce_log.zero_()
            n_log, last_log_t, last_log_tokens = 0, now, tokens_seen
        if do_ckpt:
            path = save_checkpoint("periodic")
            last_ckpt = time.time()
            if main:
                print(f"  [checkpoint saved -> {path.name}]", flush=True)
                if cfg["push"]:
                    push_checkpoint_dataset(cfg["output_dir"], cfg["dataset_id"],
                                            f"{cfg['run_id']}: step={step}, tokens={tokens_seen:,} (in progress)")
            if distributed:
                dist.barrier()
    path = save_checkpoint(stop_reason)
    loop_min = (time.time() - t_loop) / 60
    summary = {"stop_reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
               "session_tokens": tokens_seen - start_tokens, "loop_minutes": round(loop_min, 1),
               "tokens_per_sec": round((tokens_seen - start_tokens) / max(1e-6, loop_min * 60), 1),
               "world": world, "micro_bs": micro_bs, "accum": accum, "tokens_per_step": tokens_per_step,
               "skipped_steps": skipped, "grad_ckpt": model.grad_ckpt, "n_params": n_params, "phase_id": phase,
               "phase_start_step": phase_start_step,
               "final_checkpoint": path.name if path else None, "progress": round(progress, 4),
               "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if cuda else 0.0}
    if main:
        json.dump(summary, open(Path(cfg["output_dir"]) / "train_summary.json", "w"), indent=1)
        log(f"[D] stopped ({stop_reason}) at step={step:,}, tokens={tokens_seen:,} after {loop_min:.1f} min "
            f"-> {path.name}")
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    if stop_reason.startswith("50 non-finite"):
        sys.exit(3)


if __name__ == "__main__":
    train_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/c_model.py
"""c_model.py -- Model C (prefix-LM) classes, copied from p1_07_model_C.ipynb."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

decode = None          # only used by C's text generation, not needed here


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/a_model.py
"""a_model.py -- Model A (Llama-style decoder) classes, copied from p1_03_model_A.ipynb."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction -- cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def _forward_inner(self, x, cos, sin, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, cache=None, use_checkpoint=False):
        # Checkpointing needs a cache-free forward (no in-place cache
        # mutation across a recomputed backward pass) and only makes
        # sense during training, never during KV-cache generation.
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, cache)


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values."""
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: raw hidden states, no LM head. attention_mask
        is accepted for interface compatibility with later phases (e.g. a
        padded batch) but unused here -- Phase 1 pretraining always packs
        full, unpadded windows, so every position is valid.
        """
        assert mode == "causal", "MiniGPT is decoder-only; only causal encoding is supported."
        B, T = input_ids.shape
        x = self.embed(input_ids)
        cos = self.rope_cos[:T].to(x.device)
        sin = self.rope_sin[:T].to(x.device)
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, cache=None, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, caches=None):
        if caches is None:
            return self.head(self.encode(ids))
        # Generation path (KV cache present): identical to encode() but
        # threading a per-layer cache through and offsetting RoPE by how
        # much of the cache is already filled.
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache=caches[i], use_checkpoint=False)
        x = self.norm_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", top_p=0.9, temperature=1.0):
        """Contract method. `sampler`: "greedy" (argmax, deterministic --
        what the smoke-test submission uses) or "top_p" (nucleus sampling).
        Stops early (per sequence, via a done mask) at <eos> or max_seq_len.
        """
        assert kv_cache, "This notebook's generate() only implements the KV-cache path -- it's what training will actually ship."
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        # A prompt longer than the model's context has nowhere to put
        # max_new_tokens (or, if it's longer than max_seq_len outright,
        # RoPE's precomputed cos/sin tables don't even cover it -- a hard
        # crash, not a quality issue). Truncate to the most recent tokens
        # -- the ones closest to where generation continues -- rather
        # than letting this crash the whole generation loop over one long
        # dev-set example.
        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            print(f"  [generate() warning: prompt has {input_ids.shape[1]} tokens, "
                  f"truncating to the last {max_prompt_len} to leave room for "
                  f"{max_new_tokens} new tokens within max_seq_len={self.config['max_seq_len']}]")
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])

        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)  # prefill: process the whole prompt at once
        next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all():
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)  # one new token at a time
            next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        self.train()
        return generated

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        mask = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * mask
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function: the one place that turns a config dict into a
    model instance, so every later phase constructs Model A the same way.
    """
    return MiniGPT(config)


USE_GRAD_CHECKPOINTING = True
EOS_ID = None

In [ ]:
%%writefile /kaggle/working/b_model.py
"""b_model.py -- Model B (local/global decoder) classes, copied from p1_04_model_B.ipynb."""
import math
Z_LOSS_COEF = 1e-4
EOS_ID = None
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


_TRAIN_MASK_CACHE = {}


def sliding_window_mask(q_start, n_q, n_k, window, device, cacheable=False):
    """Boolean [n_q, n_k] mask, True = may attend. Query i sits at absolute
    position q_start + i; key j at position j. Allowed iff the key is not in
    the future AND is less than `window` positions back.
    Only training masks are cached (training always uses one fixed shape);
    eval/generation shapes vary per prompt, so caching them would leak memory.
    """
    cache_key = (n_q, n_k, window, str(device))
    if cacheable and cache_key in _TRAIN_MASK_CACHE:
        return _TRAIN_MASK_CACHE[cache_key]
    q_pos = torch.arange(q_start, q_start + n_q, device=device)[:, None]
    k_pos = torch.arange(n_k, device=device)[None, :]
    allowed = (k_pos <= q_pos) & (q_pos - k_pos < window)
    if cacheable:
        _TRAIN_MASK_CACHE[cache_key] = allowed
    return allowed


class LocalGlobalAttention(nn.Module):
    """MQA attention. Local layers: RoPE + sliding-window mask. Global
    layers: no RoPE (NoPE), plain causal attention over everything.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, use_rope, qk_norm):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.use_rope = use_rope
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)
        self.q_norm = RMSNorm(d_head) if qk_norm else None
        self.k_norm = RMSNorm(d_head) if qk_norm else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        # The cache keeps the single KV head (that's MQA's memory saving);
        # expand it to all query heads only for the attention call itself.
        # An explicit copy keeps SDPA on its fast memory-efficient kernel.
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            # Global layer. Training / prefill: square causal. Decode step
            # (one query): may see every cached key -> no mask needed.
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg, is_global):
        super().__init__()
        self.is_global = is_global
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = LocalGlobalAttention(cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"],
                                         use_rope=not is_global, qk_norm=cfg["qk_norm"])
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg["d_model"], cfg["d_ff"])

    def _forward_inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, attn_mask, cache)


class KVCache:
    """Pre-allocated per-layer cache. Keeps the full history (with 1 KV
    head it's only ~0.5 MB per layer at 2048 tokens); the sliding window is
    applied by the attention mask, not by evicting entries.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


def is_global_layer(layer_idx, global_every):
    return (layer_idx + 1) % global_every == 0


class LocalGlobalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config, is_global=is_global_layer(i, config["global_every"]))
            for i in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # tied embeddings

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def layer_pattern(self):
        return "".join("G" if b.is_global else "L" for b in self.blocks)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def _run_blocks(self, x, pos_offset, caches, use_ckpt):
        T = x.shape[1]
        assert pos_offset + T <= self.config["max_seq_len"], "Sequence longer than max_seq_len."
        cos = self.rope_cos[pos_offset:pos_offset + T]
        sin = self.rope_sin[pos_offset:pos_offset + T]
        local_mask = sliding_window_mask(pos_offset, T, pos_offset + T, self.config["window"], x.device,
                                         cacheable=self.training and caches is None)
        for i, blk in enumerate(self.blocks):
            mask = None if blk.is_global else local_mask
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, mask, cache=cache, use_checkpoint=use_ckpt)
        return x

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: hidden states, no LM head. attention_mask is
        accepted for interface compatibility (Phase 1 packs full windows)."""
        assert mode == "causal", "Model B is decoder-only; only causal encoding is supported."
        x = self.embed(input_ids)
        x = self._run_blocks(x, 0, None, use_ckpt=self.training and USE_GRAD_CHECKPOINTING)
        return self.norm_f(x)

    def forward(self, ids, targets=None, caches=None):
        """No targets -> logits. With targets -> tensor [1, 2] = [cross-entropy,
        cross-entropy + z-loss]. The [1, 2] shape is so nn.DataParallel can
        stack one row per GPU; callers take .mean(0).
        In training mode, fp16 autocast is switched on here (inside the
        module) so it also applies inside DataParallel's worker threads.
        """
        use_amp = self.training and ids.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            if caches is None:
                h = self.encode(ids)
            else:
                pos_offset = caches[0].pos
                assert ids.shape[1] == 1 or pos_offset == 0, "Prefill must start from an empty cache."
                h = self.norm_f(self._run_blocks(self.embed(ids), pos_offset, caches, use_ckpt=False))
            logits = self.head(h)
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        target_logit = logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        ce = (lse - target_logit).mean()           # identical to F.cross_entropy
        total = ce + Z_LOSS_COEF * lse.pow(2).mean()
        return torch.stack([ce, total]).unsqueeze(0)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy",
                 top_p=0.9, temperature=1.0, repetition_penalty=1.0):
        """Contract method. Stops per sequence at <eos>. Prompts too long to
        leave room for max_new_tokens are cut to their most recent tokens."""
        assert kv_cache, "Only the KV-cache path is implemented."
        was_training = self.training
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])
        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)[:, -1, :]
        next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)
        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all() or generated.shape[1] >= max_total:
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)[:, -1, :]
            next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)

        self.train(was_training)
        return generated

    @staticmethod
    def _penalize(logits, previous_ids, penalty):
        """CTRL-style repetition penalty on every token already present."""
        if penalty == 1.0:
            return logits
        scores = logits.gather(1, previous_ids)
        scores = torch.where(scores > 0, scores / penalty, scores * penalty)
        return logits.scatter(1, previous_ids, scores)

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        keep = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function."""
    return LocalGlobalGPT(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/pcr_lib.py
"""pcr_lib.py -- Phase 7 (Prior Case Retrieval, PCR) library.

Task: for every query judgment, rank its split's candidate pool and return the 7 most relevant prior
cases. Metric: micro-F1@7.

Pipeline (every experiment):
  CHUNK     every judgment (query or candidate) -> word chunks of CHUNK_WORDS (<= MAX_CHUNKS, spread
            evenly over the judgment); tokenised per model.
  LEXICAL   BM25 over each pool (own code). It gives (a) hard negatives for training -- highly ranked
            but not cited cases -- and (b) a lexical score fused with the model's score at the end.
  ALIGN     for each (query, cited case) pair, the chunk pairs that share the most TF-IDF weight:
            training shows the model the PARTS of two judgments that make one cite the other,
            not two random 512-token windows (evidence-aligned chunk pairs).
  TRAIN     dual encoder (one Phase-1 backbone for both sides, attention pooling over tokens,
            L2-normalised chunk vectors); InfoNCE over in-batch candidates + one BM25 hard negative
            per query, other cases cited by the same query masked out (no false negatives).
  SCORE     every chunk of every query and candidate is encoded once; three document-level
            scores: mean (one vector per judgment), max-sim (each query chunk finds its best
            candidate chunk, averaged -- late interaction at chunk level), max-pair.
  FUSE      final = z(model score) + w * z(BM25), w in [0, 1] so the model always leads; the
            aggregation and w are chosen by an honest 2-fold check on dev (choose on one half of the
            dev queries, score the other half, both ways).

Methods (own code): LoRMA (Bihany et al. 2025), QLoRA (Dettmers et al. 2023), IA3 (Liu et al. 2022),
prompt tuning (Lester et al. 2021), partial / layer-wise fine-tuning, full fine-tuning. Distillation:
half-depth D student learns the teacher's in-batch similarity distribution (listwise KD).
"""
import json
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CHUNK_WORDS = 360               # ~500 tokens
MAX_CHUNKS = 12
MAX_TOK = 512
TOPK = 7
TAU = 0.05
FUSE_W = (0.0, 0.25, 0.5, 0.75, 1.0)
AGGS = ("mean", "maxsim", "maxpair")
LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
STOP = set("the of and to in a is that be by for on as was with it this which or are at an not his her their from have has had "
           "been were said under shall may any such other these those there would could should will also no its upon all "
           "into than so if but we they he she them him who whom whose being per".split())


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_json_any(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


def _first(d, keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def _text(t):
    if isinstance(t, list):
        t = " ".join(str(x) for x in t)
    return " ".join(str(t or "").split())


def _items(obj):
    if isinstance(obj, dict):
        for k in ("data", "queries", "candidates", "documents", "docs", "items"):
            if k in obj and isinstance(obj[k], list):
                return obj[k]
        return [dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj


def parse_pool(obj):
    """Candidate pool -> {str id: {"raw_id", "text"}}."""
    out = {}
    for i, v in enumerate(_items(obj)):
        raw = _first(v, ("id", "candidate_id", "doc_id", "case_id", "name"), i)
        out[str(raw)] = {"raw_id": raw, "text": _text(_first(v, ("text", "document", "judgment", "content"), ""))}
    return out


def parse_queries(obj):
    """Queries -> list of {"id": str, "raw_id", "text", "rel": [str ids] or None}."""
    out = []
    for i, v in enumerate(_items(obj)):
        raw = _first(v, ("id", "query_id", "doc_id", "case_id", "name"), i)
        rel = _first(v, ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "citations"))
        out.append({"id": str(raw), "raw_id": raw, "text": _text(_first(v, ("text", "document", "judgment", "content"), "")),
                    "rel": None if rel is None else [str(r) for r in rel]})
    return out


def chunk_words(text, n=CHUNK_WORDS, k=MAX_CHUNKS):
    """Word chunks of n words; at most k of them, spread evenly (first and last always kept)."""
    w = text.split()
    ch = [" ".join(w[i:i + n]) for i in range(0, max(1, len(w)), n)] or [""]
    if len(ch) > k:
        idx = np.unique(np.round(np.linspace(0, len(ch) - 1, k)).astype(int))
        ch = [ch[i] for i in idx]
    return ch


# ---------------------------------------------------------------------------
# Lexical side: BM25 (own code) and TF-IDF chunk alignment
# ---------------------------------------------------------------------------
_TOK = r"(?u)\b[a-z0-9][a-z0-9]+\b"


class BM25:
    """Okapi BM25 over one pool. Query terms weighted by log(1 + tf) (queries are whole judgments)."""

    def __init__(self, texts, k1=1.2, b=0.75):
        from sklearn.feature_extraction.text import CountVectorizer
        self.vec = CountVectorizer(token_pattern=_TOK, lowercase=True, stop_words=sorted(STOP), dtype=np.float32)
        X = self.vec.fit_transform(texts).tocsr()
        n = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1])
        self.idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(X.sum(1)).ravel()
        norm = k1 * (1 - b + b * dl / max(1e-9, dl.mean()))
        X = X.tocoo()
        data = X.data * (k1 + 1) / (X.data + norm[X.row]) * self.idf[X.col]
        import scipy.sparse as sp
        self.W = sp.csr_matrix((data, (X.row, X.col)), shape=X.shape)

    def scores(self, texts):
        Q = self.vec.transform(texts).tocsr().astype(np.float32)
        Q.data = np.log1p(Q.data)
        return np.asarray((Q @ self.W.T).todense(), dtype=np.float32)


def chunk_tfidf(chunk_lists):
    """TF-IDF vectors for every chunk of every document; returns (vectoriser, {doc: sparse rows})."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    flat = [c for _, chs in chunk_lists for c in chs]
    vec = TfidfVectorizer(token_pattern=_TOK, lowercase=True, stop_words=sorted(STOP), sublinear_tf=True, min_df=2, dtype=np.float32)
    M = vec.fit_transform(flat).tocsr()
    out, k = {}, 0
    for d, chs in chunk_lists:
        out[d] = M[k:k + len(chs)]
        k += len(chs)
    return out


def best_pairs(Qm, Cm, top=3):
    """Top (query chunk, candidate chunk) pairs by TF-IDF cosine, plus, for every query chunk, the
    candidate chunk that matches it best."""
    S = np.asarray((Qm @ Cm.T).todense())
    flat = np.argsort(-S, axis=None)[:top]
    pairs = [(int(i // S.shape[1]), int(i % S.shape[1])) for i in flat]
    return pairs, S.argmax(1).astype(np.int16).tolist()


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def topk_ids(scores, cand_ids, query_id=None, k=TOPK):
    order = np.argsort(-scores, kind="mergesort")
    out = []
    for j in order:
        if cand_ids[j] != query_id:
            out.append(cand_ids[j])
            if len(out) == k:
                break
    return out


def micro_f1(preds, golds):
    """preds, golds: lists of id lists. Micro precision / recall / F1 at the size of preds."""
    tp = sum(len(set(p) & set(g)) for p, g in zip(preds, golds))
    npred = sum(len(p) for p in preds)
    ngold = sum(len(g) for g in golds)
    P, R = tp / max(1, npred), tp / max(1, ngold)
    return {"f1": round(2 * P * R / max(1e-12, P + R), 4), "precision": round(P, 4), "recall": round(R, 4)}


def zrows(S):
    S = np.asarray(S, dtype=np.float64)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)


def choose_fusion(neural, bm25, golds, cand_ids, qids, seed=0):
    """neural: {aggregation: scores [nq, nc]}. Pick (aggregation, w) by an honest 2-fold check
    over the dev queries; then fit on all dev. Returns (choice, report)."""
    nq = len(golds)
    rng = np.random.RandomState(seed)
    a = np.zeros(nq, bool)
    a[rng.permutation(nq)[: nq // 2]] = True
    zb = zrows(bm25)
    grid = {(g, w): zrows(S) + w * zb for g, S in neural.items() for w in FUSE_W}

    def f1(sel, key):
        F = grid[key]
        return micro_f1([topk_ids(F[i], cand_ids, qids[i]) for i in np.where(sel)[0]], [golds[i] for i in np.where(sel)[0]])["f1"]
    table = {f"{g}|w={w}": f1(np.ones(nq, bool), (g, w)) for g, w in grid}
    if a.any() and (~a).any():
        ka = max(grid, key=lambda k: f1(a, k))
        kb = max(grid, key=lambda k: f1(~a, k))
        preds = [None] * nq
        for sel, k in ((~a, ka), (a, kb)):
            for i in np.where(sel)[0]:
                preds[i] = topk_ids(grid[k][i], cand_ids, qids[i])
        honest = micro_f1(preds, golds)
    else:
        honest = None
    best = max(grid, key=lambda k: f1(np.ones(nq, bool), k))
    return best, {"grid_fit": table, "chosen": f"{best[0]}|w={best[1]}", "dev_2fold": honest,
                  "dev_fit": micro_f1([topk_ids(grid[best][i], cand_ids, qids[i]) for i in range(nq)], golds)}


# ---------------------------------------------------------------------------
# Adaptation methods
# ---------------------------------------------------------------------------
class IA3Out(nn.Module):
    """IA3 on keys / values: y = l * (W0 x), l = 1 at the start."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """IA3 on the feed-forward activation: y = W2 (l * a), l = 1 at the start."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def blocks_of(kind, model):
    return model.enc_blocks if kind in ("D", "Ds") else model.blocks


def final_norm(kind, model):
    return model.enc_norm if kind in ("D", "Ds") else model.norm_f


def max_len_of(kind, model):
    cfg = getattr(model, "config", {}) or {}
    return cfg.get("max_enc_len") if kind in ("D", "Ds") else cfg.get("max_seq_len")


class LoRMALinear(nn.Module):
    """LoRMA, additive-identity form: y = (I + s B A) W0 x = h + s B A h with h = W0 x.
    B = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


NF4_VALUES = (-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
              -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
              0.7229568362236023, 1.0)


def quantize_nf4(w, block=64, dq_block=256):
    """4-bit NormalFloat (QLoRA): blocks of `block` weights scaled by their absmax, each weight
    -> nearest of 16 normal quantiles (2 codes per byte). Double quantisation: the block absmax
    values are themselves stored as int8 (minus their mean, scaled per `dq_block` blocks)."""
    flat = w.detach().float().flatten()
    n = flat.numel()
    flat = F.pad(flat, (0, (-n) % (2 * block)))
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp(min=1e-12)
    table = torch.tensor(NF4_VALUES)
    idx = torch.bucketize((blocks / absmax[:, None]).flatten(), (table[1:] + table[:-1]) / 2)
    codes = ((idx[0::2] << 4) | idx[1::2]).to(torch.uint8)
    c = absmax.mean()
    a = absmax - c
    nb = a.numel()
    a = F.pad(a, (0, (-nb) % dq_block)).view(-1, dq_block)
    s2 = a.abs().amax(1).clamp(min=1e-12)
    qabs = torch.round(a / s2[:, None] * 127).clamp(-127, 127).to(torch.int8).flatten()
    return {"codes": codes, "qabs": qabs, "s2": s2.float(), "c": c.float().reshape(1), "n": n, "nb": nb,
            "shape": tuple(w.shape), "block": block, "dq_block": dq_block}


def dequantize_nf4(codes, qabs, s2, c, table, n, nb, shape, block, dq_block, dtype=torch.float32):
    idx = torch.stack([(codes >> 4).long(), (codes & 15).long()], 1).flatten()
    absmax = ((qabs.view(-1, dq_block).float() / 127) * s2[:, None]).flatten()[:nb] + c
    w = (table[idx].view(-1, block).float() * absmax[:, None]).flatten()[:n]
    return w.view(shape).to(dtype)


class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored in NF4 (4-bit) + double-quantised scales, de-quantised on
    the fly; trainable LoRA y = W_nf4 x + s B A x (B = 0, so it starts as the 4-bit model)."""

    def __init__(self, base, r, alpha, block=64, dq_block=256):
        super().__init__()
        q = quantize_nf4(base.weight, block, dq_block)
        self.out_features, self.in_features = base.out_features, base.in_features
        for k in ("codes", "qabs", "s2", "c"):
            self.register_buffer(k, q[k])
        self.register_buffer("table", torch.tensor(NF4_VALUES))
        self.meta = (q["n"], q["nb"], q["shape"], block, dq_block)
        with torch.no_grad():                       # how far the 4-bit weight is from the Phase-1 weight
            w = base.weight.detach().float()
            self._err = float((self.weight() - w).norm() / w.norm().clamp(min=1e-12))
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def weight(self, dtype=torch.float32):
        return dequantize_nf4(self.codes, self.qabs, self.s2, self.c, self.table, *self.meta, dtype=dtype)

    def forward(self, x):
        dt = torch.float16 if x.is_cuda else torch.float32
        if self.training:                           # training: de-quantise on the fly (only 4-bit kept)
            self._wcache = None
            W = self.weight(dt)
        else:                                       # generation: de-quantise once, reuse at every step
            if getattr(self, "_wcache", None) is None or self._wcache.dtype != dt or self._wcache.device != x.device:
                self._wcache = self.weight(dt)
            W = self._wcache
        y = F.linear(x, W, self.bias)
        return y + F.linear(F.linear(x, self.A), self.B) * self.scale


class PrefixTuner(nn.Module):
    """Prefix tuning, deep form (Li & Liang 2021; P-Tuning v2): P extra positions at the front of
    every chunk whose hidden state at the INPUT OF EVERY BLOCK is a learned vector. Every real token
    can attend to them at every layer (their keys/values come from the learned states through the
    frozen projections). Implemented with forward pre-hooks, so no model code changes."""

    def __init__(self, blocks, n_prefix, d):
        super().__init__()
        self.P = n_prefix
        self.prefix = nn.Parameter(torch.zeros(len(blocks), n_prefix, d))
        self.capture = None
        self.handles = [b.register_forward_pre_hook(self._hook(i)) for i, b in enumerate(blocks)]

    def _hook(self, i):
        def hook(module, args):
            x = args[0]
            if self.capture is not None:            # initialisation pass: record real activations
                v = x[:, self.P:2 * self.P].detach().float().mean(0)
                if v.shape[0] < self.P:
                    v = torch.cat([v, v.mean(0, keepdim=True).expand(self.P - v.shape[0], -1)])
                self.capture[i] = v
                return None
            p = self.prefix[i].to(x.dtype).unsqueeze(0).expand(x.shape[0], -1, -1)
            return (torch.cat([p, x[:, self.P:]], 1),) + tuple(args[1:])
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def norm_params(model):
    return [p for m in model.modules() if type(m).__name__.endswith("Norm") for p in m.parameters(recurse=False)]

def storage_mb(model):
    """Deployable backbone size: fp16 parameters + packed 4-bit buffers (QLoRA)."""
    b = sum(p.numel() * 2 for p in model.parameters())
    b += sum(buf.numel() * buf.element_size() for n, buf in model.named_buffers() if n.split(".")[-1] in ("codes", "qabs", "s2", "c"))
    return round(b / 2 ** 20, 1)


class AttnPool(nn.Module):
    """Attention pooling over tokens; the query starts at 0, i.e. as mean pooling."""

    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(d))

    def forward(self, h, mask):
        s = (h @ self.q) / math.sqrt(h.shape[-1])
        w = torch.softmax(s.masked_fill(~mask, -1e4), -1)
        return (w.unsqueeze(-1) * h).sum(1)


class ChunkEncoder(nn.Module):
    """token chunk [B, T] (right-padded) -> chunk vector [B, d]."""

    def __init__(self, kind, model, d, tuner=None, prefix_id=0):
        super().__init__()
        self.kind, self.m, self.d = ("D" if kind == "Ds" else kind), model, d
        self.tuner, self.prefix_id = tuner, prefix_id
        self.P = tuner.P if tuner is not None else 0
        self.max_len = max_len_of(kind, model)
        self.pool = AttnPool(d)

    def states(self, ids, mask):
        P = self.P
        if P:
            if self.max_len and ids.shape[1] + P > self.max_len:
                ids, mask = ids[:, :self.max_len - P], mask[:, :self.max_len - P]
            B = ids.shape[0]
            ids = torch.cat([ids.new_full((B, P), self.prefix_id), ids], 1)
            full = torch.cat([mask.new_ones((B, P)), mask], 1)
        else:
            full = mask
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=ids.is_cuda):
            if self.kind == "D":
                h = self.m.encode(ids, full)
            elif self.kind == "C":
                h = self.m.encode(ids, mode="prefix", prefix_len=full.sum(1))
            else:                                   # A, B: left to right
                h = self.m.encode(ids)
        h = h.float()
        return (h[:, P:], mask) if P else (h, mask)

    def forward(self, ids, mask):
        h, m = self.states(ids, mask)
        return self.pool(h, m)

    @torch.no_grad()
    def init_prefix(self, ids, mask):
        """Li & Liang: start the prefix from activations of real words (here: the first P tokens
        of real training chunks, at every layer)."""
        if self.tuner is None:
            return
        was = self.training
        self.eval()
        self.tuner.capture = {}
        self.states(ids, mask)
        for i, v in self.tuner.capture.items():
            self.tuner.prefix.data[i] = v.to(self.tuner.prefix.dtype)
        self.tuner.capture = None
        self.train(was)




def apply_method(model, kind, method, exp):
    """Freeze the Phase-1 model, then insert / open what the method trains.
    Returns (trainable backbone parameters, PrefixTuner or None)."""
    for p in model.parameters():
        p.requires_grad_(False)
    tuner = None
    if method == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    elif method == "partial":
        blocks = list(blocks_of(kind, model))
        for b in blocks[-exp["top_k"]:]:
            for p in b.parameters():
                p.requires_grad_(True)
        for p in final_norm(kind, model).parameters():
            p.requires_grad_(True)
    elif method in ("lorma", "qlora"):
        names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in LINEAR_TARGETS]
        assert names, "no target linear layers found"
        for n in names:
            lin = model.get_submodule(n)
            _replace(model, n, LoRMALinear(lin, exp.get("r", 16), exp.get("alpha", 32)) if method == "lorma"
                     else QLoRALinear(lin, exp.get("r", 16), exp.get("alpha", 32)))
    elif method == "ia3":
        for n, m in list(model.named_modules()):
            last = n.split(".")[-1]
            if isinstance(m, nn.Linear) and last in ("k_proj", "v_proj"):
                _replace(model, n, IA3Out(m))
            elif isinstance(m, nn.Linear) and last == "down_proj":
                _replace(model, n, IA3In(m))
    elif method == "prompt":                       # prompt tuning: learned states at the INPUT only (block 0)
        tuner = PrefixTuner([list(blocks_of(kind, model))[0]], exp.get("n_prompt", 20), exp["d_model"])
    else:
        raise ValueError(method)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return n + (tuner.prefix.numel() if tuner is not None else 0), tuner


def pad_tokens(seqs, pad_id, device):
    L = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(np.asarray(s, dtype=np.int64))
        mask[i, :len(s)] = True
    return ids.to(device), mask.to(device)


def embed(enc, seqs, pad_id, device):
    ids, mask = pad_tokens(seqs, pad_id, device)
    return F.normalize(enc(ids, mask).float(), dim=-1)


def lr_mult(p, warm=0.05):
    return min(1.0, p / warm) if p < warm else max(0.1, 1 - (p - warm) / (1 - warm))


def trainable_state(enc):
    return {k: v for k, v in enc.named_parameters() if v.requires_grad}


def build_encoder(exp, make_backbone, device, pad_id):
    model, d, kind = make_backbone(exp["model"])
    n_backbone = sum(p.numel() for p in model.parameters())
    n_tr, tuner = apply_method(model, kind, exp["method"], dict(exp, d_model=d))
    enc = ChunkEncoder(kind, model, d, tuner, pad_id).to(device)
    return enc, kind, n_backbone, n_tr


# ---------------------------------------------------------------------------
# Training: InfoNCE over evidence-aligned chunk pairs (+ listwise KD)
# ---------------------------------------------------------------------------
def sample_batch(examples, chunks, B, rng, hard_neg=True):
    """examples: [{"q", "pos": {cand: [(qi, cj), ...]}, "neg": {cand: [best cand chunk per query chunk]}, "rel": set}]."""
    qs, cs, owner, pos_col, rels = [], [], [], [], []
    for ex in rng.sample(examples, min(B, len(examples))):
        p = rng.choice(sorted(ex["pos"]))
        qi, pj = rng.choice(ex["pos"][p])
        qs.append(chunks[ex["q"]][qi])
        pos_col.append(len(cs))
        cs.append(chunks[p][pj])
        owner.append(p)
        if hard_neg and ex["neg"]:
            n = rng.choice(sorted(ex["neg"]))
            best = ex["neg"][n]
            cs.append(chunks[n][best[min(qi, len(best) - 1)]])
            owner.append(n)
        rels.append(ex["rel"])
    mask = torch.zeros(len(qs), len(cs), dtype=torch.bool)       # True = another cited case: not a negative
    for i, r in enumerate(rels):
        for j, o in enumerate(owner):
            if j != pos_col[i] and o in r:
                mask[i, j] = True
    return qs, cs, torch.tensor(pos_col), mask


def contrastive_step(enc, qs, cs, pos, mask, pad, device, teacher=None, T=2.0, alpha=0.0):
    zq, zc = embed(enc, qs, pad, device), embed(enc, cs, pad, device)
    S = (zq @ zc.t()) / TAU
    S = S.masked_fill(mask.to(device), -1e4)
    loss = F.cross_entropy(S, pos.to(device))
    if teacher is not None and alpha > 0:
        with torch.no_grad():
            St = (embed(teacher, qs, pad, device) @ embed(teacher, cs, pad, device).t() / TAU).masked_fill(mask.to(device), -1e4)
        kd = F.kl_div(F.log_softmax(S / T, -1), F.softmax(St / T, -1), reduction="batchmean") * T * T
        loss = (1 - alpha) * loss + alpha * kd
    return loss


@torch.no_grad()
def val_loss(enc, batches, pad, device):
    enc.eval()
    return float(np.mean([float(contrastive_step(enc, *b, pad, device)) for b in batches]))


def train(exp, data, enc, device, log, teacher=None):
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    ex = data["train_examples"]
    rng.shuffle(ex)
    n_val = max(4, len(ex) // 10)
    val_ex, tr_ex = ex[:n_val], ex[n_val:]
    vrng = random.Random(0)
    val_batches = [sample_batch(val_ex, data["chunks"], exp["batch_q"], vrng) for _ in range(exp.get("val_batches", 12))]  # always with hard negatives
    params = [p for p in enc.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": params, "lr": exp["lr"], "base_lr": exp["lr"], "weight_decay": 0.01}])
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    steps = exp["steps"]
    budget = exp["budget_min"] * 60
    T, alpha = exp.get("kd_T", 2.0), exp.get("kd_alpha", 0.0)
    log(f"[{exp['name']}] {len(tr_ex)} training queries ({sum(len(e['pos']) for e in tr_ex)} cited pairs), {n_val} validation queries | "
        f"{exp['batch_q']} queries x (cited{' + BM25 hard negative' if exp.get('hard_neg', True) else ', in-batch negatives only'}) per step, {steps} steps planned, budget {exp['budget_min']} min"
        + (f" | KD alpha {alpha} T {T}" if teacher is not None else ""))
    t0, step, progress = time.time(), 0, 0.0
    st = {"best": float("inf"), "state": None, "hist": [], "run": 0.0, "n": 0}
    evals = [0.25, 0.5, 0.75, 1.0]

    def evaluate():
        v = val_loss(enc, val_batches, data["pad"], device)
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run"] / max(1, st["n"]), 4),
                           "val_loss": round(v, 4), "minutes": round((time.time() - t0) / 60, 1)})
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | train loss {st['hist'][-1]['train_loss']:.4f} | "
            f"validation loss {v:.4f} | {st['hist'][-1]['minutes']} min")
        st["run"], st["n"] = 0.0, 0
        if v < st["best"]:
            st["best"] = v
            st["state"] = {k: p.detach().cpu().clone() for k, p in trainable_state(enc).items()}

    while progress < 1.0:
        enc.train()
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * lr_mult(progress)
        b = sample_batch(tr_ex, data["chunks"], exp["batch_q"], rng, exp.get("hard_neg", True))
        loss = contrastive_step(enc, *b, data["pad"], device, teacher, T, alpha)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        st["run"], st["n"], step = st["run"] + float(loss.detach()), st["n"] + 1, step + 1
        progress = min(1.0, max(step / steps, (time.time() - t0) / budget))
        while evals and progress >= evals[0]:
            evals.pop(0)
            evaluate()
    enc.load_state_dict(st["state"], strict=False)
    return {"train_minutes": round((time.time() - t0) / 60, 1), "steps": step, "history": st["hist"], "best_val_loss": st["best"],
            "peak_mem_train_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}


# ---------------------------------------------------------------------------
# Encoding and document-level scores
# ---------------------------------------------------------------------------
@torch.no_grad()
def encode_docs(enc, ids, chunks, pad, device, bs):
    """-> (vectors [n_docs, MAX_CHUNKS, d] fp16, mask [n_docs, MAX_CHUNKS])."""
    enc.eval()
    items = [(i, j) for i, d in enumerate(ids) for j in range(len(chunks[d]))]
    items.sort(key=lambda t: -len(chunks[ids[t[0]]][t[1]]))
    V = torch.zeros(len(ids), MAX_CHUNKS, enc.d, dtype=torch.float16)
    M = torch.zeros(len(ids), MAX_CHUNKS, dtype=torch.bool)
    for k in range(0, len(items), bs):
        part = items[k:k + bs]
        z = embed(enc, [chunks[ids[i]][j] for i, j in part], pad, device).half().cpu()
        for (i, j), v in zip(part, z):
            V[i, j], M[i, j] = v, True
    return V, M


@torch.no_grad()
def doc_scores(Vq, Mq, Vc, Mc, device, block=None):
    """{aggregation: [n_queries, n_candidates]} -- mean vector, max-sim (late interaction over chunks), max pair."""
    if block is None:                               # keep the [block, K, n_cand, K] similarity tensor near 600 MB
        block = int(max(1, min(64, 1.5e8 // (Vc.shape[1] ** 2 * max(1, len(Vc))))))
    Vc_d, Mc_d = Vc.to(device).float(), Mc.to(device)
    mc = F.normalize((Vc_d * Mc_d[..., None]).sum(1) / Mc_d.sum(1, keepdim=True).clamp(min=1), dim=-1)
    out = {a: [] for a in AGGS}
    nc, K, d = Vc.shape
    flat_c = Vc_d.reshape(nc * K, d)
    for s in range(0, len(Vq), block):
        vq, mq = Vq[s:s + block].to(device).float(), Mq[s:s + block].to(device)
        mqv = F.normalize((vq * mq[..., None]).sum(1) / mq.sum(1, keepdim=True).clamp(min=1), dim=-1)
        out["mean"].append((mqv @ mc.t()).cpu())
        sim = (vq.reshape(-1, d) @ flat_c.t()).reshape(len(vq), K, nc, K)
        sim = sim.masked_fill(~Mc_d[None, None], -2).masked_fill(~mq[:, :, None, None], -2)
        best = sim.max(-1).values                                                 # [b, Kq, nc]
        cnt = mq.sum(1, keepdim=True).clamp(min=1).float()
        out["maxsim"].append(((best * mq[..., None]).sum(1) / cnt).cpu())
        out["maxpair"].append(best.max(1).values.cpu())
    return {a: torch.cat(v).numpy() for a, v in out.items()}


def run_experiment(exp, data, make_backbone, device, log, out_dir, teacher_dir=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    enc, kind, n_backbone, n_tr = build_encoder(exp, make_backbone, device, data["pad"])
    if enc.tuner is not None:                       # prompt from real token embeddings (Lester et al.)
        sample = sorted((data["chunks"][e["q"]][0] for e in data["train_examples"][:200]), key=len)[-8:]
        enc.init_prefix(*pad_tokens(sample, data["pad"], device))
    sto = storage_mb(enc.m)
    log(f"[{exp['name']}] {exp['title']} | backbone {exp['model']} {n_backbone / 1e6:.1f}M ({sto} MB stored) | trainable {n_tr:,} "
        f"({100 * n_tr / n_backbone:.2f}%) + {sum(p.numel() for p in enc.pool.parameters()):,} pooling")
    info = {"exp": exp, "n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": round(100 * n_tr / n_backbone, 3), "storage_mb": sto}
    if exp["method"] == "qlora":
        info["nf4_relative_weight_error"] = round(float(np.mean([m._err for m in enc.m.modules() if isinstance(m, QLoRALinear)])), 4)
    dev, test = data["splits"]["dev"], data["splits"]["test"]

    def score_split(sp, tag):
        t0 = time.time()
        Vq, Mq = encode_docs(enc, sp["query_ids"], data["chunks"], data["pad"], device, exp["infer_bs"])
        Vc, Mc = encode_docs(enc, sp["cand_ids"], data["chunks"], data["pad"], device, exp["infer_bs"])
        secs = time.time() - t0
        return doc_scores(Vq, Mq, Vc, Mc, device), secs

    def report(neural, tag):
        choice, rep = choose_fusion(neural, dev["bm25"], dev["golds"], dev["cand_raw"], dev["query_raw"], exp["seed"])
        neural_only = {g: micro_f1([topk_ids(S[i], dev["cand_raw"], dev["query_raw"][i]) for i in range(len(S))], dev["golds"])["f1"]
                       for g, S in neural.items()}
        rep["neural_only_fit"] = neural_only
        log(f"[{exp['name']}] {tag}: model alone (F1@7) " + ", ".join(f"{g} {v:.4f}" for g, v in neural_only.items()) +
            f" | fused, chosen {rep['chosen']}: 2-fold {rep['dev_2fold']['f1'] if rep['dev_2fold'] else None} / fit {rep['dev_fit']['f1']:.4f}")
        return choice, rep

    if exp.get("zero_shot"):                        # the Phase-1 model before any retrieval training
        zs, _ = score_split(dev, "dev")
        _, info["dev_zero_shot"] = report(zs, "zero-shot (no training)")
    teacher = None
    if teacher_dir is not None:
        teacher, _, _, _ = build_encoder(dict(exp["teacher"], model="D"), make_backbone, device, data["pad"])
        state = torch.load(Path(teacher_dir) / "trainable.pt", map_location="cpu", weights_only=False)
        teacher.load_state_dict(state, strict=False)
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        info["teacher_params"] = sum(p.numel() for p in teacher.m.parameters())
    info.update(train(exp, data, enc, device, log, teacher))
    if teacher is not None:
        del teacher
        torch.cuda.empty_cache() if device.type == "cuda" else None
    if exp.get("save_trainable"):
        torch.save({k: v.detach().cpu() for k, v in trainable_state(enc).items()}, out_dir / "trainable.pt")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    dev_neural, dev_secs = score_split(dev, "dev")
    choice, rep = report(dev_neural, "dev")
    info.update({"dev": rep, "dev_encode_minutes": round(dev_secs / 60, 1),
                 "docs_per_s": round((len(dev["query_ids"]) + len(dev["cand_ids"])) / max(1e-6, dev_secs), 2),
                 "peak_mem_infer_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0})
    g, w = choice
    dev_final = zrows(dev_neural[g]) + w * zrows(dev["bm25"])
    info["dev_analysis"] = analysis(dev_final, dev)
    test_neural, _ = score_split(test, "test")
    test_final = zrows(test_neural[g]) + w * zrows(test["bm25"])
    write_submission(out_dir, test, test_final)
    np.savez_compressed(out_dir / "scores.npz", dev=dev_final.astype(np.float32), test=test_final.astype(np.float32))
    json.dump(info, open(out_dir / "metrics.json", "w"), indent=1, default=str)
    log(f"[{exp['name']}] DONE dev F1@7 {rep['dev_2fold']['f1'] if rep['dev_2fold'] else None} (2-fold) / {rep['dev_fit']['f1']:.4f} (fit) "
        f"P {rep['dev_fit']['precision']:.4f} R {rep['dev_fit']['recall']:.4f} -> {out_dir / 'submission.zip'}")
    return info


def analysis(final, sp):
    """Where the system fails: F1@7 by number of cited cases and by query length; rank of cited cases."""
    preds = [topk_ids(final[i], sp["cand_raw"], sp["query_raw"][i]) for i in range(len(final))]
    golds = sp["golds"]
    out = {"by_cited_count": [], "by_query_length": []}
    n = np.array([len(g) for g in golds])
    for lo, hi, name in ((1, 3, "1-3"), (4, 7, "4-7"), (8, 15, "8-15"), (16, 10 ** 6, "16+")):
        m = (n >= lo) & (n <= hi)
        if m.any():
            out["by_cited_count"].append({"cited": name, "queries": int(m.sum()),
                                          **micro_f1([preds[i] for i in np.where(m)[0]], [golds[i] for i in np.where(m)[0]])})
    L = np.array(sp["query_words"])
    qs = np.quantile(L, [0.33, 0.67])
    for k, (lo, hi) in enumerate(((0, qs[0]), (qs[0], qs[1]), (qs[1], np.inf))):
        m = (L >= lo) & (L < hi) if k < 2 else (L >= lo)
        if m.any():
            out["by_query_length"].append({"tercile": k + 1, "words_from": int(lo), "queries": int(m.sum()),
                                           "f1": micro_f1([preds[i] for i in np.where(m)[0]], [golds[i] for i in np.where(m)[0]])["f1"]})
    pos = {c: j for j, c in enumerate(sp["cand_raw"])}
    ranks = []
    for i, g in enumerate(golds):
        order = np.argsort(-final[i], kind="mergesort")
        r = np.empty(len(order), int)
        r[order] = np.arange(1, len(order) + 1)
        ranks += [int(r[pos[c]]) for c in g if c in pos]
    ranks = np.array(ranks)
    if not len(ranks):
        return out
    out["cited_case_rank"] = {"median": float(np.median(ranks)), "in_top7": float(np.mean(ranks <= 7)),
                              "in_top20": float(np.mean(ranks <= 20)), "in_top100": float(np.mean(ranks <= 100))}
    return out


def write_submission(out_dir, sp, final):
    import zipfile
    out_dir = Path(out_dir)
    pred, out_of = {}, dict(zip(sp["cand_raw"], sp["cand_out"]))
    for i, q in enumerate(sp["query_raw"]):
        top = topk_ids(final[i], sp["cand_raw"], q)
        assert len(top) == TOPK and len(set(top)) == TOPK
        pred[str(sp["query_out"][i])] = [out_of[c] for c in top]
    assert len(pred) == len(sp["query_raw"])
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(pred, f)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")


# ---------------------------------------------------------------------------
# Self-tests (tiny models, CPU, seconds)
# ---------------------------------------------------------------------------
def self_test(make_tiny, device, kinds=("D", "C", "A", "B")):
    torch.manual_seed(0)
    rng = np.random.RandomState(0)
    a, b = list(rng.randint(3, 60, 11)), list(rng.randint(3, 60, 5))
    for kind in kinds:
        m0, d = make_tiny(kind)
        ref = ChunkEncoder(kind, m0, d).to(device).eval()
        before = embed(ref, [a, b], 0, device)
        assert (embed(ref, [b], 0, device)[0] - before[1]).abs().max() < 1e-4, f"{kind}: padding changes the vector"
        for method in ("lorma", "qlora", "ia3", "partial", "full", "prompt"):
            m, _ = make_tiny(kind)
            m.load_state_dict(m0.state_dict())
            n_tr, tuner = apply_method(m, kind, method, {"r": 4, "alpha": 8, "top_k": 1, "n_prompt": 3, "d_model": d})
            enc = ChunkEncoder(kind, m, d, tuner).to(device).eval()
            enc.pool.load_state_dict(ref.pool.state_dict())
            after = embed(enc, [a, b], 0, device)
            if method in ("lorma", "ia3", "partial", "full"):
                assert (after - before).abs().max() < 1e-4, f"{kind}/{method} does not start as the Phase-1 model"
            elif method == "qlora":
                assert F.cosine_similarity(after, before, -1).min() > 0.95, f"{kind}/qlora too far from the Phase-1 model"
            else:
                assert (embed(enc, [b], 0, device)[0] - after[1]).abs().max() < 1e-4, f"{kind}/prompt: padding changes the vector"
                enc.train()
                embed(enc, [a, b], 0, device).sum().backward()
                assert tuner.prefix.grad is not None and tuner.prefix.grad.abs().sum() > 0
                assert all(p.grad is None for p in m.parameters()), "prompt tuning must keep the backbone frozen"
            if method == "ia3":
                assert n_tr > 0 and all(n.endswith(".l") for n, p in m.named_parameters() if p.requires_grad)
    # contrastive step + listwise KD run and mask cited cases
    m, d = make_tiny(kinds[0])
    enc = ChunkEncoder(kinds[0], m, d).to(device)
    chunks = {f"q{i}": [list(rng.randint(3, 60, 9)) for _ in range(3)] for i in range(4)}
    chunks.update({f"c{i}": [list(rng.randint(3, 60, 7)) for _ in range(2)] for i in range(8)})
    exs = [{"q": f"q{i}", "pos": {f"c{i}": [(0, 1), (2, 0)], f"c{(i + 1) % 4}": [(1, 1)]}, "neg": {f"c{i + 4}": [0, 1, 1]},
            "rel": {f"c{i}", f"c{(i + 1) % 4}"}} for i in range(4)]
    qs, cs, pos, mask = sample_batch(exs, chunks, 4, random.Random(1))
    assert len(qs) == 4 and len(cs) == 8 and mask.sum() > 0 and not mask[torch.arange(4), pos].any()
    loss = contrastive_step(enc, qs, cs, pos, mask, 0, device, teacher=enc, alpha=0.5)
    loss.backward()
    assert torch.isfinite(loss)
    # document scores: chunk padding is invisible; aggregations agree with a direct computation
    Vq = F.normalize(torch.randn(3, MAX_CHUNKS, 8), dim=-1).half()
    Vc = F.normalize(torch.randn(5, MAX_CHUNKS, 8), dim=-1).half()
    Mq = torch.zeros(3, MAX_CHUNKS, dtype=torch.bool)
    Mc = torch.zeros(5, MAX_CHUNKS, dtype=torch.bool)
    Mq[:, :2], Mq[0, :4], Mc[:, :3] = True, True, True
    Mc[1] = False
    Mc[1, :1] = True
    S = doc_scores(Vq, Mq, Vc, Mc, device, block=2)
    q, c = Vq[0, :4].float(), Vc[1, :1].float()
    assert abs(S["maxsim"][0, 1] - float((q @ c.t()).max(1).values.mean())) < 1e-3
    assert abs(S["maxpair"][0, 1] - float((q @ c.t()).max())) < 1e-3
    assert abs(S["mean"][0, 1] - float(F.normalize(q.mean(0), dim=0) @ F.normalize(c.mean(0), dim=0))) < 1e-3
    # metric, BM25, fusion choice
    assert micro_f1([["a", "b"], ["c", "d"]], [["a"], ["c", "x", "y"]]) == {"f1": round(2 * 0.5 * 0.5 / 1.0, 4), "precision": 0.5, "recall": 0.5}
    bm = BM25(["murder appeal dismissed", "land acquisition compensation", "murder conviction appeal"])
    s = bm.scores(["appeal against murder conviction"])[0]
    assert s.argmax() == 2 and s[1] == 0
    assert topk_ids(np.array([3.0, 2, 1, 0, -1, -2, -3, -4]), list("qabcdefg"), "q") == list("abcdefg")
    ids = [f"c{i}" for i in range(10)]
    neural = {"mean": rng.randn(6, 10), "maxsim": rng.randn(6, 10), "maxpair": rng.randn(6, 10)}
    golds = [[ids[i]] for i in range(6)]
    choice, rep = choose_fusion(neural, rng.randn(6, 10), golds, ids, [f"q{i}" for i in range(6)])
    assert choice[0] in AGGS and choice[1] in FUSE_W and rep["dev_2fold"] is not None
    print("pcr_lib self-test passed: LoRMA / IA3 / partial / full start as the Phase-1 model, QLoRA as its 4-bit copy, prompt "
          f"tuning trains only the prompt and is padding-safe ({', '.join(kinds)}); contrastive + KD step with cited cases masked; "
          "chunk-level scores (mean / max-sim / max-pair) exact and padding-safe; BM25; micro-F1@7; 2-fold fusion choice.")

In [ ]:
%%writefile /kaggle/working/pcr_train.py
"""pcr_train.py -- one Phase-7 experiment: python pcr_train.py <config.json> <experiment> <gpu index or -1>"""
import importlib
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import pcr_lib as PL  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
TOK = {"A": "T1", "B": "T1", "C": "T2", "D": "T2", "Ds": "T2"}[exp["model"]]
data = torch.load(cfg["data"][TOK], weights_only=False)


def make_backbone(kind):
    """Phase-1 model from its checkpoint -> (module, d_model, kind)."""
    if kind in ("D", "Ds"):
        import legal_d
        ck = torch.load(cfg["ckpt"]["D"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        m.dec_blocks = torch.nn.ModuleList()          # encoder only
        if kind == "Ds":                              # distillation student: every other encoder layer (DistilBERT-style)
            m.enc_blocks = torch.nn.ModuleList([m.enc_blocks[i] for i in exp["student_layers"] if i < len(m.enc_blocks)])
        m.grad_ckpt = exp.get("grad_ckpt", True)
        return m, ck["config"]["d_model"], kind
    mod = importlib.import_module({"A": "a_model", "B": "b_model", "C": "c_model"}[kind])
    ck = torch.load(cfg["ckpt"][kind], map_location="cpu", weights_only=False)
    m = mod.build_model(ck["config"])
    m.load_state_dict(ck["model_state"])
    mod.USE_GRAD_CHECKPOINTING = exp.get("grad_ckpt", True)
    return m, ck["config"]["d_model"], kind


def log(*a):
    print(*a, flush=True)


teacher_dir = Path(cfg["out_dir"]) / exp["teacher_run"] if exp.get("teacher_run") else None
PL.run_experiment(exp, data, make_backbone, device, log, Path(cfg["out_dir"]) / name, teacher_dir)

## 3. Data: download, check, chunk, BM25, hard negatives, evidence-aligned pairs

In [ ]:
sys.path.insert(0, str(WORK_DIR))
import pcr_lib as PL  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"


def hf_file(*names):
    # first of the candidate paths that exists in the repo (the capstone doc names the first one)
    last = None
    for n in names:
        try:
            return hf_hub_download(REPO, n, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:      # noqa: BLE001
            last = e
    try:
        from huggingface_hub import list_repo_files
        have = [f for f in list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN) if "7" in f.split("/")[0]]
    except Exception:               # noqa: BLE001
        have = "?"
    raise FileNotFoundError(f"none of {names} in {REPO} ({last}). Phase-7 files in the repo: {have}")


t0 = time.time()
FILES = {"train": ("Train/train", "Train/train_candidates"), "dev": ("Dev/dev", "Dev/dev_candidates"),
         "test": ("Test/test", "Test/test_candidates")}
QUERIES, POOLS = {}, {}
for s, (q, c) in FILES.items():
    qobj = PL.load_json_any(hf_file(f"Phase_7_PCR/{q}.json", f"Phase_7_PCR/{q}.jsonl"))
    cobj = PL.load_json_any(hf_file(f"Phase_7_PCR/{c}.json", f"Phase_7_PCR/{c}.jsonl"))
    if s == "train":
        one = qobj[0] if isinstance(qobj, list) else next(iter(qobj.values()))
        print("query fields:", list(one.keys()) if isinstance(one, dict) else type(one))
    seen, qs = set(), []
    for x in PL.parse_queries(qobj):                # one prediction per query id
        if x["id"] not in seen:
            seen.add(x["id"])
            qs.append(x)
    pool = PL.parse_pool(cobj)
    # ids are IndianKanoon numbers: if a file writes them without the zero padding, match on the number
    norm = {c.lstrip("0") or "0": c for c in pool}
    fix = lambda i: i if i in pool else norm.get(i.lstrip("0") or "0", i)
    n_fixed = 0
    for x in qs:
        x["self"] = fix(x["id"])                    # the query's own id in the pool (never retrieved for itself)
        if x["rel"] is not None:
            new = [fix(r) for r in x["rel"]]
            n_fixed += sum(a != b for a, b in zip(new, x["rel"]))
            x["rel"] = new
    if n_fixed:
        print(f"{s}: {n_fixed} cited ids matched to the pool by number (zero padding differed)")
    QUERIES[s], POOLS[s] = qs, pool
    del qobj, cobj
print(f"downloaded + parsed in {time.time() - t0:.0f}s")
REPORT = {}
for s in FILES:
    qs, pool = QUERIES[s], POOLS[s]
    for what, texts in (("queries", [x["text"] for x in qs]), ("candidates", [v["text"] for v in pool.values()])):
        empty = sum(not t for t in texts)
        assert empty <= 0.05 * len(texts), f"{s} {what}: {empty} have no text -- check the field names"
    qw = np.array([len(x["text"].split()) for x in qs])
    cw = np.array([len(v["text"].split()) for v in pool.values()])
    r = {"queries": len(qs), "pool": len(pool), "query_words_median": float(np.median(qw)), "cand_words_median": float(np.median(cw)),
         "queries_in_own_pool": sum(x["self"] in pool for x in qs)}
    if s != "test":
        assert all(x["rel"] is not None for x in qs), f"{s}: queries need 'relevant_candidates'"
        nrel = np.array([len(set(x["rel"])) for x in qs])
        miss = sum(c not in pool for x in qs for c in set(x["rel"]))
        r.update({"cited_per_query_mean": float(nrel.mean()), "cited_per_query_median": float(np.median(nrel)),
                  "cited_max": int(nrel.max()), "queries_citing_none": int((nrel == 0).sum()),
                  "cited_ids_missing_from_pool": int(miss), "cited_share_of_pool_used": round(len({c for x in qs for c in x["rel"]}) / max(1, len(pool)), 3)})
    REPORT[s] = r
    print(f"{s}: " + ", ".join(f"{k} {v:.1f}" if isinstance(v, float) else f"{k} {v}" for k, v in r.items()))

# ---- chunks, BM25, evidence-aligned training pairs ------------------------------------------------
t0 = time.time()
CH_TEXT = {}                                       # internal id "<split>:q|c:<id>" -> list of chunk texts
SPLITS = {}
for s in FILES:
    qs, pool = QUERIES[s], POOLS[s]
    cids = list(pool)
    bm = PL.BM25([pool[c]["text"] for c in cids])
    B = bm.scores([x["text"] for x in qs])
    SPLITS[s] = {"query_ids": [f"{s}:q:{x['id']}" for x in qs], "cand_ids": [f"{s}:c:{c}" for c in cids],
                 "query_raw": [x["self"] for x in qs], "cand_raw": cids, "query_out": [x["raw_id"] for x in qs],
                 "cand_out": [pool[c]["raw_id"] for c in cids], "golds": [sorted(set(x["rel"] or [])) for x in qs],
                 "bm25": B, "query_words": [len(x["text"].split()) for x in qs]}
    if s != "train":
        for x in qs:
            CH_TEXT[f"{s}:q:{x['id']}"] = PL.chunk_words(x["text"])
        for c in cids:
            CH_TEXT[f"{s}:c:{c}"] = PL.chunk_words(pool[c]["text"])
print(f"BM25 for all splits in {time.time() - t0:.0f}s")

N_NEG = 10                                          # hard negatives kept per query (one is sampled per step)
tr, pool = SPLITS["train"], POOLS["train"]
col = {c: j for j, c in enumerate(tr["cand_raw"])}
plan = []
for i, x in enumerate(QUERIES["train"]):
    pos = [c for c in tr["golds"][i] if c in col and c != x["self"]]
    if not pos:
        continue
    order = np.argsort(-tr["bm25"][i], kind="mergesort")
    neg = [tr["cand_raw"][j] for j in order[:N_NEG + len(pos) + 1] if tr["cand_raw"][j] not in set(pos) | {x["self"]}][:N_NEG]
    plan.append((x, pos, neg))
used = {c for _, p, n in plan for c in p + n}
for x, _, _ in plan:
    CH_TEXT[f"train:q:{x['id']}"] = PL.chunk_words(x["text"])
for c in used:
    CH_TEXT[f"train:c:{c}"] = PL.chunk_words(pool[c]["text"])
TF = PL.chunk_tfidf([(k, v) for k, v in CH_TEXT.items() if k.startswith("train:")])
TRAIN_EX = []
for x, pos, neg in plan:
    qk = f"train:q:{x['id']}"
    TRAIN_EX.append({"q": qk, "pos": {f"train:c:{c}": PL.best_pairs(TF[qk], TF[f"train:c:{c}"])[0] for c in pos},
                     "neg": {f"train:c:{c}": PL.best_pairs(TF[qk], TF[f"train:c:{c}"])[1] for c in neg},
                     "rel": {f"train:c:{c}" for c in pos}})
del TF
nch = np.array([len(v) for v in CH_TEXT.values()])
print(f"training: {len(TRAIN_EX)} queries, {sum(len(e['pos']) for e in TRAIN_EX)} (query, cited case) pairs, {N_NEG} BM25 hard "
      f"negatives each | {len(CH_TEXT)} documents chunked, {nch.mean():.1f} chunks each (cap {PL.MAX_CHUNKS}) | {time.time() - t0:.0f}s")

# ---- BM25 alone on dev: the bar every model has to clear -------------------------------------------
dv = SPLITS["dev"]
bm_pred = [PL.topk_ids(dv["bm25"][i], dv["cand_raw"], dv["query_raw"][i]) for i in range(len(dv["golds"]))]
BM25_DEV = PL.micro_f1(bm_pred, dv["golds"])
for k in (20, 100):
    hit = [len(set(PL.topk_ids(dv["bm25"][i], dv["cand_raw"], dv["query_raw"][i], k)) & set(g)) for i, g in enumerate(dv["golds"])]
    BM25_DEV[f"recall@{k}"] = round(sum(hit) / max(1, sum(len(g) for g in dv["golds"])), 4)
print("BM25 alone, dev:", BM25_DEV)
REPORT.update({"bm25_dev": BM25_DEV, "train_queries_used": len(TRAIN_EX), "chunks_per_doc": float(nch.mean())})
del SPLITS["train"], QUERIES, POOLS
torch.save({"dev": SPLITS["dev"], "test": SPLITS["test"], "bm25_dev": BM25_DEV}, OUT_DIR / "p7_meta.pt")
gc.collect()

## 4. Tokenize (T2 for C and D, T1 for A and B)

In [ ]:
import legal_d  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

legal_d.load_tokenizer(T2_PATH)
sp = legal_d.sp
t1 = Tokenizer.from_file(str(T1_PATH))


def enc_t2(texts):
    try:
        return sp.encode(texts, out_type=int, num_threads=os.cpu_count() or 4)
    except TypeError:
        return sp.encode(texts, out_type=int)


def enc_t1(texts):
    return [e.ids for e in t1.encode_batch(texts)]


SPEC = {"T2": (enc_t2, sp.pad_id() if sp.pad_id() >= 0 else 0, sp.get_piece_size()),
        "T1": (enc_t1, t1.token_to_id("<pad>") or 0, t1.get_vocab_size())}
keys = list(CH_TEXT)
flat = [c for k in keys for c in CH_TEXT[k]]
DATA_PATH = {}
for name, (enc, pad, vocab) in SPEC.items():
    t0 = time.time()
    assert vocab <= 65536, "token ids must fit uint16"
    ids = []
    for s in range(0, len(flat), 20000):           # bounded memory
        ids += [np.asarray(x[:PL.MAX_TOK], dtype=np.uint16) for x in enc(flat[s:s + 20000])]
    it = iter(ids)
    chunks = {k: [next(it) for _ in CH_TEXT[k]] for k in keys}
    ntok = np.array([len(x) for x in ids])
    print(f"{name}: {len(ids)} chunks tokenized in {time.time() - t0:.0f}s | tokens per chunk: median {np.median(ntok):.0f}, "
          f"cut at {PL.MAX_TOK}: {100 * np.mean(ntok >= PL.MAX_TOK):.1f}%")
    REPORT[f"{name}_tokens_per_chunk_median"] = float(np.median(ntok))
    DATA_PATH[name] = str(OUT_DIR / f"p7_data_{name}.pt")
    torch.save({"tokenizer": name, "pad": pad, "chunks": chunks, "train_examples": TRAIN_EX, "splits": SPLITS}, DATA_PATH[name])
    del ids, chunks
json.dump(REPORT, open(OUT_DIR / "data_report.json", "w"), indent=1, default=str)
del CH_TEXT, flat
gc.collect()

## 5. Self-tests (tiny models, seconds)

In [ ]:
import a_model  # noqa: E402
import b_model  # noqa: E402
import c_model  # noqa: E402

for _m in (a_model, b_model, c_model):
    _m.USE_GRAD_CHECKPOINTING = True


def make_tiny(kind):
    if kind == "D":
        cfg = dict(legal_d.MODEL_D_CONFIG, vocab_size=64, d_model=32, n_enc_layer=2, n_dec_layer=1, n_head=4,
                   n_kv_head=2, d_head=8, d_ff=48, max_enc_len=64, max_dec_len=64)
        m = legal_d.build_model(cfg).eval()
        m.dec_blocks = torch.nn.ModuleList()
        return m, 32
    if kind == "C":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=64,
                   qk_norm=True, bias=True, mlp="gelu", tokenizer_id="T2", attn_type="prefix", tied_embeddings=True)
        return c_model.build_model(cfg).eval(), 32
    if kind == "A":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=128)
        return a_model.build_model(cfg).eval(), 32
    if kind == "B":
        cfg = dict(vocab_size=64, d_model=32, n_layer=3, n_head=4, n_kv_head=1, d_head=8, d_ff=48, max_seq_len=128,
                   window=4, global_every=3, qk_norm=True, tokenizer_id="T1", attn_type="local+global", tied_embeddings=True)
        return b_model.build_model(cfg).eval(), 32
    raise KeyError(kind)


PL.self_test(make_tiny, torch.device("cpu"), kinds=("D", "C", "A", "B"))

## 6. Experiment settings

`budget_min` caps training; `est_min` (training + encoding dev and test pools) is only used to skip a run that could not finish in the session.

In [ ]:
COMMON = dict(seed=13, grad_ckpt=True, val_batches=12)
EXPERIMENTS = {
    "S1": dict(title="D encoder + LoRMA", model="D", method="lorma", r=16, alpha=32, lr=5e-4, batch_q=12, steps=1500,
               budget_min=35, infer_bs=64, save_trainable=True, zero_shot=True, est_min=75, submit=True),
    "S2": dict(title="C + IA3", model="C", method="ia3", lr=3e-3, batch_q=12, steps=1500, budget_min=35, infer_bs=48,
               est_min=70, submit=True),
    "S6": dict(title="KD student: half of D <- S1", model="Ds", method="full", student_layers=[0, 2, 4, 6, 8, 10], lr=1e-4,
               batch_q=12, steps=1500, budget_min=35, infer_bs=96, teacher_run="S1", teacher=dict(method="lorma", r=16, alpha=32),
               kd_alpha=0.5, kd_T=2.0, est_min=70, submit=True),
    "S3": dict(title="B + QLoRA (NF4 4-bit + LoRA)", model="B", method="qlora", r=16, alpha=32, lr=5e-4, batch_q=8, steps=1500,
               budget_min=40, infer_bs=32, est_min=90, submit=True),
    "S4": dict(title="A + partial FT (top 4 blocks)", model="A", method="partial", top_k=4, lr=1e-4, batch_q=8, steps=1500,
               budget_min=40, infer_bs=24, est_min=100, submit=True),
    "S5": dict(title="D encoder + prompt tuning (20 vectors)", model="D", method="prompt", n_prompt=20, lr=1e-2, batch_q=12,
               steps=1500, budget_min=30, infer_bs=64, est_min=65, submit=True),
}
EXPERIMENTS["R1"] = dict(EXPERIMENTS["S1"], title="ablation: S1 without BM25 hard negatives (in-batch only)", hard_neg=False,
                         save_trainable=False, zero_shot=False, budget_min=30, est_min=55, submit=False)
RUN_ORDER = [("S1", []), ("S2", []), ("S6", ["S1"]), ("S3", []), ("S4", []), ("S5", []), ("R1", [])]
TIME_LIMIT_MIN = 11 * 60          # Kaggle stops at 12 h: never start a run that would end after 11 h
CFG = {"work_dir": str(WORK_DIR), "data": DATA_PATH, "ckpt": CKPT, "out_dir": str(OUT_DIR), "common": COMMON,
       "experiments": EXPERIMENTS}
CONFIG_PATH = WORK_DIR / "pcr_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
for name, e in EXPERIMENTS.items():
    print(f"{name}: {e['title']}")

## 7. Run (two at a time, one per GPU; S6 waits for its teacher S1)

In [ ]:
def stream(proc, tag):
    with open(OUT_DIR / f"log_{tag}.txt", "w") as f:
        for line in proc.stdout:
            f.write(line)
            print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
while queue or running:
    for slot in slots:
        if slot in running:
            continue
        ready = [q for q in queue if all(d in codes for d in q[1])]
        if not ready:
            continue
        name, deps = ready[0]
        queue.remove(ready[0])
        if any(codes.get(d) != 0 for d in deps):
            print(f">>> skipping {name}: {deps} did not all finish", flush=True)
            codes[name] = -1
            continue
        used = (time.time() - NB_T0) / 60
        if used + EXPERIMENTS[name]["est_min"] > TIME_LIMIT_MIN:
            print(f">>> skipping {name}: {used:.0f} min used, it needs ~{EXPERIMENTS[name]['est_min']} more", flush=True)
            codes[name] = -2
            continue
        env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        proc = subprocess.Popen([sys.executable, str(WORK_DIR / "pcr_train.py"), str(CONFIG_PATH), name, str(slot)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        th = threading.Thread(target=stream, args=(proc, name), daemon=True)
        th.start()
        running[slot] = (name, proc, th)
        print(f">>> started {name} ({EXPERIMENTS[name]['title']}) on {'GPU ' + str(slot) if slot >= 0 else 'CPU'} "
              f"({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)

## 8. S7 ensemble + BM25 reference

In [ ]:
# S7: mean of the z-scored final scores (model + BM25, as each run chose) of every finished run
META = torch.load(OUT_DIR / "p7_meta.pt", weights_only=False)
dev, test = META["dev"], META["test"]
members = [n for n, _ in RUN_ORDER if EXPERIMENTS[n]["submit"] and (OUT_DIR / n / "scores.npz").exists()]
if len(members) >= 2:
    Z = {n: np.load(OUT_DIR / n / "scores.npz") for n in members}
    dev7 = np.mean([PL.zrows(Z[n]["dev"]) for n in members], 0)
    test7 = np.mean([PL.zrows(Z[n]["test"]) for n in members], 0)
    S7 = OUT_DIR / "S7"
    S7.mkdir(exist_ok=True)
    pred = [PL.topk_ids(dev7[i], dev["cand_raw"], dev["query_raw"][i]) for i in range(len(dev7))]
    m7 = PL.micro_f1(pred, dev["golds"])
    # leave-one-out: what each member adds to the ensemble
    loo = {}
    for n in members:
        rest = [k for k in members if k != n]
        if rest:
            d = np.mean([PL.zrows(Z[k]["dev"]) for k in rest], 0)
            loo[n] = PL.micro_f1([PL.topk_ids(d[i], dev["cand_raw"], dev["query_raw"][i]) for i in range(len(d))], dev["golds"])["f1"]
    PL.write_submission(S7, test, test7)
    json.dump({"exp": {"name": "S7", "title": "Ensemble of " + " + ".join(members)}, "dev_f1": m7, "leave_one_out_f1": loo,
               "dev_analysis": PL.analysis(dev7, dev)}, open(S7 / "metrics.json", "w"), indent=1)
    print(f"S7 ensemble of {members}: dev F1@7 {m7['f1']:.4f} P {m7['precision']:.4f} R {m7['recall']:.4f}")
    print("  without each member:", {k: round(v, 4) for k, v in loo.items()})
print("BM25 alone, dev:", META["bm25_dev"])

## 9. Results, analysis tables, submissions

In [ ]:
rows, by = [], {}
for name in [n for n, _ in RUN_ORDER] + ["S7"]:
    mp = OUT_DIR / name / "metrics.json"
    if not mp.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mp))
    if name == "S7":
        r = {"run": "S7", "title": m["exp"]["title"], "model": "-", "technique": "ensemble", "trainable": 0, "trainable_pct": 0,
             "storage_mb": 0, "dev_f1_2fold": None, "dev_f1": m["dev_f1"]["f1"], "dev_p": m["dev_f1"]["precision"],
             "dev_r": m["dev_f1"]["recall"], "model_alone_best": None, "chosen": "mean of z-scores", "train_min": 0,
             "docs_per_s": 0, "peak_gb": 0}
    else:
        e, d = EXPERIMENTS[name], m["dev"]
        r = {"run": name, "title": e["title"], "model": e["model"], "technique": e["method"] if name != "S6" else "KD + full",
             "trainable": m["n_trainable"], "trainable_pct": m["trainable_pct"], "storage_mb": m["storage_mb"],
             "dev_f1_2fold": d["dev_2fold"]["f1"] if d["dev_2fold"] else None, "dev_f1": d["dev_fit"]["f1"],
             "dev_p": d["dev_fit"]["precision"], "dev_r": d["dev_fit"]["recall"],
             "model_alone_best": max(d["neural_only_fit"].values()), "chosen": d["chosen"], "train_min": m["train_minutes"],
             "docs_per_s": m["docs_per_s"], "peak_gb": max(m.get("peak_mem_train_gb", 0), m.get("peak_mem_infer_gb", 0))}
    rows.append(r)
    by[name] = (r, m)
assert rows, "No run finished -- read the error lines above."
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
fmt = lambda v: "   -  " if v is None else f"{v:6.4f}"
print(f"{'run':4s} {'title':36s} {'trainable':>12s} {'%':>6s} {'MB':>5s} {'F1 2f':>6s} {'F1':>6s} {'P':>6s} {'R':>6s} {'alone':>6s} "
      f"{'chosen':>16s} {'train':>6s} {'doc/s':>6s} {'GB':>5s}")
for r in rows:
    print(f"{r['run']:4s} {r['title'][:36]:36s} {r['trainable']:>12,} {r['trainable_pct']:6.2f} {r['storage_mb']:5.0f} {fmt(r['dev_f1_2fold'])} "
          f"{r['dev_f1']:6.4f} {r['dev_p']:6.4f} {r['dev_r']:6.4f} {fmt(r['model_alone_best'])} {r['chosen'][:16]:>16s} {r['train_min']:6.1f} "
          f"{r['docs_per_s']:6.1f} {r['peak_gb']:5.1f}")
b = META["bm25_dev"]
print(f"     {'BM25 alone (own code)':36s} {'':>12s} {'':>6s} {'':>5s} {'':>6s} {b['f1']:6.4f} {b['precision']:6.4f} {b['recall']:6.4f}"
      f"   (recall@20 {b['recall@20']}, recall@100 {b['recall@100']})")
print("\nF1 2f = honest 2-fold dev score (aggregation + BM25 weight chosen on the other half); F1 = choice fit on all dev; "
      "alone = best model-only score (no BM25)")

print("\ndocument representation (model alone, dev F1@7):")
for name, (r, m) in by.items():
    if name != "S7":
        print(f"  {name}: " + " | ".join(f"{g} {v:.4f}" for g, v in m["dev"]["neural_only_fit"].items()))
print("\nBM25 weight w for the chosen aggregation (dev F1@7, fit):")
for name, (r, m) in by.items():
    if name != "S7":
        g = m["dev"]["chosen"].split("|")[0]
        print(f"  {name} ({g}): " + " | ".join(f"w={k.split('w=')[1]} {v:.4f}" for k, v in m["dev"]["grid_fit"].items() if k.startswith(g + "|")))
if "S1" in by and "dev_zero_shot" in by["S1"][1]:
    z = by["S1"][1]["dev_zero_shot"]
    print(f"\ntransfer of Phase-1 representations (D encoder, no retrieval training): model alone "
          + ", ".join(f"{g} {v:.4f}" for g, v in z["neural_only_fit"].items()) + f" | fused {z['dev_fit']['f1']:.4f}"
          f"  ->  after LoRMA: model alone {by['S1'][0]['model_alone_best']:.4f} | fused {by['S1'][0]['dev_f1']:.4f}")
if "R1" in by and "S1" in by:
    print(f"\nhard negatives (D + LoRMA): in-batch + 1 BM25 hard negative per query F1@7 {by['S1'][0]['dev_f1']} (model alone "
          f"{by['S1'][0]['model_alone_best']}) vs in-batch only F1@7 {by['R1'][0]['dev_f1']} (model alone {by['R1'][0]['model_alone_best']})")
if "S6" in by and "S1" in by:
    s6, s1 = by["S6"], by["S1"]
    print(f"\ndistillation: teacher S1 {s1[1]['n_backbone'] / 1e6:.1f}M, {s1[0]['docs_per_s']} docs/s, F1@7 {s1[0]['dev_f1']} | student S6 "
          f"{s6[1]['n_backbone'] / 1e6:.1f}M ({100 * s6[1]['n_backbone'] / s1[1]['n_backbone']:.0f}%), {s6[0]['docs_per_s']} docs/s, "
          f"F1@7 {s6[0]['dev_f1']}")
if "S3" in by and by["S3"][1].get("nf4_relative_weight_error") is not None:
    print(f"QLoRA: B stored in {by['S3'][1]['storage_mb']} MB; 4-bit weights {100 * by['S3'][1]['nf4_relative_weight_error']:.1f}% "
          f"from Phase-1 weights")
for name, (r, m) in by.items():
    if "history" in m:
        print(f"{name} training: " + ", ".join(f"{h['progress']}: {h['train_loss']}/{h['val_loss']}" for h in m["history"]))
best = max(rows, key=lambda r: r["dev_f1_2fold"] if r["dev_f1_2fold"] is not None else r["dev_f1"])
an = by[best["run"]][1]["dev_analysis"]
print(f"\nfailure analysis ({best['run']}): F1@7 by number of cited cases: "
      + " | ".join(f"{x['cited']}: {x['f1']:.3f} ({x['queries']} q)" for x in an["by_cited_count"]))
print("  by query length: " + " | ".join(f"tercile {x['tercile']} (>= {x['words_from']} words): {x['f1']:.3f}" for x in an["by_query_length"]))
if "cited_case_rank" in an:
    print("  rank of cited cases:", an["cited_case_rank"])
with zipfile.ZipFile(OUT_DIR / "report_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file() and f.suffix in (".json", ".csv", ".txt", ".log") and f.name != "prediction.json":
            z.write(f, arcname=str(f.relative_to(OUT_DIR)))
print(f"\nAll tables for the report: {OUT_DIR / 'report_files.zip'}")
print("\nSubmit these (Output tab), best first:")
pool = set(map(str, test["cand_out"]))
for r in sorted(rows, key=lambda r: -(r["dev_f1_2fold"] if r["dev_f1_2fold"] is not None else r["dev_f1"])):
    zf = OUT_DIR / r["run"] / "submission.zip"
    if zf.exists():
        with zipfile.ZipFile(zf) as z:
            assert z.namelist() == ["prediction.json"]
            pred = json.loads(z.read("prediction.json"))
        assert set(pred) == {str(q) for q in test["query_out"]}, "every test query, once"
        assert all(len(v) == 7 and len(set(map(str, v))) == 7 and set(map(str, v)) <= pool for v in pred.values())
        tag = "" if r["run"] == "S7" or EXPERIMENTS[r["run"]]["submit"] else " (ablation -- optional extra submission)"
        print(f"  {r['run']} ({r['title']}){tag}: {zf}  [dev F1@7 {r['dev_f1']:.4f}" + (f", 2-fold {r['dev_f1_2fold']:.4f}" if r["dev_f1_2fold"] else "") + "]")